# Dataset

In [ ]:
import shutil
import pandas as pd

src = '/kaggle/input/datasets/mdshajib53/breakhis-split/breakhis_split.csv'
dst = '/kaggle/working/breakhis_split.csv'

shutil.copy(src, dst)
df = pd.read_csv(dst)
print(df.shape)

# 5-fold - CausalMorph-ViT

In [ ]:
#!/usr/bin/env python3
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CausalMorph-ViT  —  Breast Histopathology  |  5-Fold CV            ║
# ║  Backbone : ViT-Tiny patch16/224  (192-dim, ~5.7M params)           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── pip installs (run in a separate cell first on Kaggle) ─────────────────
# !pip install timm scikit-learn scikit-image opencv-python-headless matplotlib seaborn ptflops

import os, hashlib, random, warnings, time, copy
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import timm
from scipy import ndimage as ndi
from skimage import measure, morphology, filters
from skimage.color import rgb2hed
from sklearn.metrics import (
    classification_report, confusion_matrix, balanced_accuracy_score,
    roc_auc_score, roc_curve, auc, matthews_corrcoef,
    cohen_kappa_score, precision_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

try:
    from ptflops import get_model_complexity_info
    HAS_PTFLOPS = True
except ImportError:
    HAS_PTFLOPS = False

warnings.filterwarnings("ignore")

# ── Paper-standard sky-blue palette ───────────────────────────────────────
SKY_BLUE   = "#87CEEB"
DEEP_BLUE  = "#1E90FF"
NAVY       = "#1a3a5c"
LIGHT_GRAY = "#f0f4f8"
PLOT_STYLE = {
    "axes.facecolor":  LIGHT_GRAY,
    "figure.facecolor": "white",
    "axes.edgecolor":  NAVY,
    "axes.labelcolor": NAVY,
    "xtick.color":     NAVY,
    "ytick.color":     NAVY,
    "text.color":      NAVY,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "font.family":     "DejaVu Sans",
}
plt.rcParams.update(PLOT_STYLE)

# ───────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ───────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ───────────────────────────────────────────────────────────────────────────
# GPU SETUP
# ───────────────────────────────────────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"{'─'*55}")
print(f"  Available GPUs : {NUM_GPUS}")
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i} : {p.name}  ({p.total_memory // 1024**2} MB)")
print(f"{'─'*55}\n")

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
CSV_PATH  = "/kaggle/working/breakhis_split.csv"
ROI_DIR   = Path("/kaggle/working/roi_patches_cm")
CACHE_DIR = Path("/kaggle/working/nmt_cache")
OUT_DIR   = Path("/kaggle/working")

TRAIN_BATCH = 32 * max(NUM_GPUS, 1)
INFER_BATCH = 32
NUM_WORKERS = min(4, os.cpu_count() or 2)

EPOCHS        = 45
WARMUP_EPOCHS = 5
N_FOLDS       = 5

IMG_SIZE = 224
STRIDE   = 112
TOP_K    = 2
TOP_CAND = 8

# NMT
N_NMT_TOKENS    = 8
NMT_FEAT_DIM    = 12
NMT_SIG_DIM     = 6
MIN_NUCLEUS_AREA = 40
MAX_NUCLEUS_AREA = 3500

# Stain (ASD)
STAIN_FEAT_DIM = 6
LAMBDA_ASD     = 0.10

# MoE
N_EXPERTS    = 4
MOE_TOP_K    = 2
MOE_MAG_DIM  = 32
LAMBDA_MOE   = 0.01

# Model
EMBED_DIM    = 192
HEAD_DROPOUT = 0.40
WEIGHT_DECAY = 5e-4
PROJ_DIM     = 192

# Loss
TEMPERATURE  = 0.07
LAMBDA_MTCL  = 0.30
LAMBDA_PROTO = 0.10
FOCAL_GAMMA  = 2.0

REFERRAL_PERCENTILE = 90
MC_T                = 30
GRL_MAX_ITER        = EPOCHS * 200

CURRICULUM = [
    {"epochs": (0,  15), "mags": {3}},
    {"epochs": (15, 30), "mags": {1, 2, 3}},
    {"epochs": (30, 45), "mags": {0, 1, 2, 3}},
]

SUBCLASS_NAMES = {
    0: "adenosis",       1: "fibroadenoma",    2: "phyllodes_tumor",
    3: "tubular_adenoma",4: "ductal_carcinoma", 5: "lobular_carcinoma",
    6: "mucinous_carcinoma", 7: "papillary_carcinoma",
}
BENIGN_CLASSES    = {0, 1, 2, 3}
MALIGNANT_CLASSES = {4, 5, 6, 7}
BINARY_NAMES      = ["benign", "malignant"]

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ROI_DIR.mkdir(parents=True, exist_ok=True)

print(f"  Train batch  : {TRAIN_BATCH}")
print(f"  Epochs       : {EPOCHS}")
print(f"  N Folds      : {N_FOLDS}")
print(f"  NMT tokens   : {N_NMT_TOKENS} × {NMT_FEAT_DIM}-dim → proj to {EMBED_DIM}-dim")
print(f"  Stain dim    : {STAIN_FEAT_DIM}  (Macenko 2×3 matrix)")
print(f"  MoE experts  : {N_EXPERTS}, top-{MOE_TOP_K} routing")
print(f"  AMP          : {'ON' if torch.cuda.is_available() else 'OFF'}\n")


# ═══════════════════════════════════════════════════════════════════════════
# [NMT]  NUCLEAR MORPHOMETRY EXTRACTOR
# ═══════════════════════════════════════════════════════════════════════════
class NuclearMorphometryExtractor:
    def __init__(self,
                 n_tokens: int = N_NMT_TOKENS,
                 img_size: int = IMG_SIZE,
                 min_area: int = MIN_NUCLEUS_AREA,
                 max_area: int = MAX_NUCLEUS_AREA):
        self.n_tokens     = n_tokens
        self.img_size     = img_size
        self.min_area     = min_area
        self.max_area     = max_area
        self.n_patches_1d = img_size // 16
        self.n_patches    = self.n_patches_1d ** 2
        self.img_area     = img_size * img_size

    def macenko_stain_params(self, img_np: np.ndarray) -> np.ndarray:
        I = img_np.astype(np.float64) / 255.0
        I = np.maximum(I, 1e-6)
        OD = -np.log(I)
        OD_flat = OD.reshape(-1, 3)
        mask = OD_flat.sum(axis=1) > 0.15
        if mask.sum() < 10:
            return np.zeros(6, dtype=np.float32)
        OD_hat = OD_flat[mask]
        try:
            _, _, V = np.linalg.svd(OD_hat - OD_hat.mean(0), full_matrices=False)
            V = V[:2, :]
            That = OD_hat @ V.T
            phi  = np.arctan2(That[:, 1], That[:, 0])
            v1   = V.T @ np.array([np.cos(np.percentile(phi, 99)),
                                    np.sin(np.percentile(phi, 99))])
            v2   = V.T @ np.array([np.cos(np.percentile(phi, 1)),
                                    np.sin(np.percentile(phi, 1))])
            v1 = np.abs(v1); v2 = np.abs(v2)
            if v1[2] < v1[1]:
                v1, v2 = v2, v1
            v1 /= (np.linalg.norm(v1) + 1e-8)
            v2 /= (np.linalg.norm(v2) + 1e-8)
            return np.stack([v1, v2], axis=0).flatten().astype(np.float32)
        except Exception:
            return np.zeros(6, dtype=np.float32)

    def extract(self, pil_img: Image.Image):
        img_np = np.array(pil_img.resize((self.img_size, self.img_size)))
        stain_params = self.macenko_stain_params(img_np)

        img_f = img_np.astype(np.float32) / 255.0
        hed   = rgb2hed(img_f)
        H_ch  = hed[:, :, 0]

        thresh = filters.threshold_otsu(H_ch)
        binary = H_ch > thresh
        binary = ndi.binary_fill_holes(binary)
        binary = morphology.remove_small_objects(binary, min_size=self.min_area)

        labeled = measure.label(binary)
        props   = measure.regionprops(labeled, intensity_image=H_ch)
        valid   = [p for p in props if self.min_area <= p.area <= self.max_area]
        valid.sort(key=lambda p: p.area * p.mean_intensity, reverse=True)

        if len(valid) > 1:
            cents = np.array([p.centroid for p in valid])
            diff  = cents[:, None] - cents[None, :]
            D     = np.sqrt((diff ** 2).sum(-1))
            np.fill_diagonal(D, np.inf)
            nnd   = D.min(1)
        else:
            nnd   = np.zeros(max(len(valid), 1))

        nuclear_density = len(valid) / self.img_area

        all_feats = []
        for i, p in enumerate(valid):
            coords = p.coords
            H_vals = H_ch[coords[:, 0], coords[:, 1]]
            std_H  = float(H_vals.std())
            peri   = max(p.perimeter, 1.0)
            circ   = 4 * np.pi * p.area / (peri ** 2)
            feat = np.array([
                p.area / self.img_area,
                peri / self.img_size,
                p.eccentricity,
                p.solidity,
                float(p.mean_intensity),
                std_H,
                float(np.clip(circ, 0, 1)),
                p.equivalent_diameter / self.img_size,
                p.extent,
                float(nnd[i]) / self.img_size,
                nuclear_density * 1000,
                float(p.euler_number),
            ], dtype=np.float32)
            all_feats.append(feat)

        top_feats  = all_feats[:self.n_tokens]
        nmt_tokens = np.zeros((self.n_tokens, NMT_FEAT_DIM), dtype=np.float32)
        for k, f in enumerate(top_feats):
            nmt_tokens[k] = np.clip(f, -5.0, 5.0)

        heatmap = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        for p in valid:
            cy, cx = int(p.centroid[0]), int(p.centroid[1])
            if 0 <= cy < self.img_size and 0 <= cx < self.img_size:
                heatmap[cy, cx] += p.area * p.mean_intensity

        if heatmap.max() > 0:
            heatmap = cv2.GaussianBlur(heatmap, (21, 21), 5)
            heatmap = heatmap / heatmap.max()
        else:
            heatmap = np.ones((self.img_size, self.img_size), dtype=np.float32) / self.n_patches

        n = self.n_patches_1d
        patch_heatmap = cv2.resize(heatmap, (n, n),
                                   interpolation=cv2.INTER_LINEAR).flatten()

        if all_feats:
            sig_mat = np.stack([f[[0, 2, 3, 4, 5, 6]] for f in all_feats[:50]])
            nmt_sig = sig_mat.mean(0)
        else:
            nmt_sig = np.zeros(NMT_SIG_DIM, dtype=np.float32)

        return (nmt_tokens.astype(np.float32),
                patch_heatmap.astype(np.float32),
                nmt_sig.astype(np.float32),
                stain_params)


_nmt_ext = NuclearMorphometryExtractor()


# ═══════════════════════════════════════════════════════════════════════════
# NMT CACHE
# ═══════════════════════════════════════════════════════════════════════════
def _df_hash(df: pd.DataFrame) -> str:
    raw = "".join(df["path"].astype(str).tolist())
    return hashlib.md5(raw.encode()).hexdigest()[:10]


class NMTCache:
    def __init__(self, df: pd.DataFrame, ext: NuclearMorphometryExtractor,
                 tag: str = ""):
        cache_key  = _df_hash(df)
        cache_path = CACHE_DIR / f"nmt_{tag}_{cache_key}.npz"

        if cache_path.exists():
            print(f"  📦 Loading NMT cache [{tag}] : {cache_path.name}")
            data            = np.load(str(cache_path))
            self.tokens     = data["tokens"]
            self.heatmaps   = data["heatmaps"]
            self.signatures = data["signatures"]
            self.stains     = data["stains"]
        else:
            print(f"  ⏳ Precomputing NMT [{tag}] for {len(df)} images …")
            tok_l, heat_l, sig_l, stain_l = [], [], [], []
            for i, (_, row) in enumerate(df.iterrows()):
                if (i + 1) % 1000 == 0 or (i + 1) == len(df):
                    print(f"     [{i+1}/{len(df)}]")
                try:
                    img = Image.open(row["path"]).convert("RGB")
                    tok, heat, sig, stain = ext.extract(img)
                except Exception:
                    tok   = np.zeros((ext.n_tokens, NMT_FEAT_DIM), dtype=np.float32)
                    heat  = np.ones(ext.n_patches, dtype=np.float32) / ext.n_patches
                    sig   = np.zeros(NMT_SIG_DIM,   dtype=np.float32)
                    stain = np.zeros(STAIN_FEAT_DIM, dtype=np.float32)
                tok_l.append(tok); heat_l.append(heat)
                sig_l.append(sig); stain_l.append(stain)

            self.tokens     = np.stack(tok_l).astype(np.float32)
            self.heatmaps   = np.stack(heat_l).astype(np.float32)
            self.signatures = np.stack(sig_l).astype(np.float32)
            self.stains     = np.stack(stain_l).astype(np.float32)
            np.savez_compressed(str(cache_path),
                                tokens=self.tokens,
                                heatmaps=self.heatmaps,
                                signatures=self.signatures,
                                stains=self.stains)
            print(f"  ✅ NMT cache saved → {cache_path.name}")

    def __getitem__(self, idx):
        return (self.tokens[idx], self.heatmaps[idx],
                self.signatures[idx], self.stains[idx])


# ═══════════════════════════════════════════════════════════════════════════
# LOAD CSV  — merge all splits into one pool for 5-fold
# ═══════════════════════════════════════════════════════════════════════════
df_all = pd.read_csv(CSV_PATH).reset_index(drop=True)
print(f"✅ CSV loaded → Total images: {len(df_all)}")
print(f"   Subclass distribution:\n{df_all['subclass_label'].value_counts().sort_index()}\n")


# ═══════════════════════════════════════════════════════════════════════════
# ESAP + NMT-DENSITY ROI EXTRACTION  (for train subsets only)
# ═══════════════════════════════════════════════════════════════════════════
from skimage.color import rgb2hed as _rgb2hed

def esap_nmt_score(patch_np: np.ndarray) -> float:
    pf  = patch_np.astype(np.float32) / 255.0
    hed = _rgb2hed(pf)
    h, e = hed[:, :, 0], hed[:, :, 1]
    esap = (float(np.mean(h > 0.10))
          + float(np.var(h))
          + 0.5 * float(np.var(e)))
    thresh  = filters.threshold_otsu(h) if h.std() > 0.01 else 0.1
    density = float((h > thresh).mean())
    return esap + 0.30 * density


def apply_roi_to_df(in_df: pd.DataFrame, fold_tag: str) -> pd.DataFrame:
    fold_roi_dir = ROI_DIR / fold_tag
    fold_roi_dir.mkdir(parents=True, exist_ok=True)
    new_rows, total = [], len(in_df)
    rng = random.Random(SEED)
    print(f"\n⏳ ESAP+NMT ROI extraction [{fold_tag}] from {total} images …")
    for i, (_, row) in enumerate(in_df.iterrows()):
        if (i + 1) % 500 == 0 or (i + 1) == total:
            print(f"   [{i+1}/{total}]")
        try:
            img  = Image.open(row["path"]).convert("RGB")
            w, h = img.size
            patches = []
            for y in range(0, h - IMG_SIZE + 1, STRIDE):
                for x in range(0, w - IMG_SIZE + 1, STRIDE):
                    patch = img.crop((x, y, x + IMG_SIZE, y + IMG_SIZE))
                    score = esap_nmt_score(np.array(patch))
                    patches.append((score, patch))
            if not patches:
                continue
            patches.sort(key=lambda t: t[0], reverse=True)
            pool = [p for _, p in patches[:TOP_CAND]]
            rois = rng.sample(pool, min(TOP_K, len(pool)))
        except Exception as e:
            print(f"   ⚠️  {row['path']}: {e}"); continue
        stem = Path(row["path"]).stem
        for k, roi in enumerate(rois):
            sp = fold_roi_dir / f"{stem}_roi{k}.png"
            roi.save(str(sp))
            new_rows.append({
                "path": str(sp), "subclass_label": row["subclass_label"],
                "binary_label": row["binary_label"],
                "mag_label": row["mag_label"], "split": "train",
            })
    roi_df    = pd.DataFrame(new_rows)
    augmented = pd.concat([in_df, roi_df], ignore_index=True)
    print(f"✅ ROI done → orig:{len(in_df)} ROI:{len(roi_df)} total:{len(augmented)}")
    return augmented


# ═══════════════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════════════
class BreakHisDataset(Dataset):
    def __init__(self, df, nmt_cache: NMTCache, transform=None):
        self.df        = df.reset_index(drop=True)
        self.nmt_cache = nmt_cache
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        tok, heat, sig, stain = self.nmt_cache[idx]
        return (img,
                int(row["subclass_label"]),
                int(row["binary_label"]),
                int(row["mag_label"]),
                torch.from_numpy(tok),
                torch.from_numpy(heat),
                torch.from_numpy(sig),
                torch.from_numpy(stain))


class BreakHisTTADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        return (img, int(row["subclass_label"]),
                int(row["binary_label"]), int(row["mag_label"]))


# ───────────────────────────────────────────────────────────────────────────
# TRANSFORMS
# ───────────────────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tta_transforms = [
    transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomVerticalFlip(p=1), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(p=1), transforms.RandomVerticalFlip(p=1), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation((15,15)), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((224,224)), transforms.RandomRotation((-15,-15)), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((256,256)), transforms.CenterCrop(224), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((224,224)), transforms.ColorJitter(0.1,0.1), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((300,300)), transforms.CenterCrop(224), transforms.ToTensor(), _norm]),
    transforms.Compose([transforms.Resize((192,192)), transforms.Pad(16), transforms.ToTensor(), _norm]),
]


# ───────────────────────────────────────────────────────────────────────────
# LOADER HELPERS
# ───────────────────────────────────────────────────────────────────────────
def _make_loader(ds, batch_size, shuffle):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=False,
                      prefetch_factor=2 if NUM_WORKERS > 0 else None,
                      drop_last=shuffle)


class _SubNMTCache:
    def __init__(self, parent: NMTCache, indices: np.ndarray):
        self.parent = parent; self.indices = indices
    def __getitem__(self, i):
        return self.parent[self.indices[i]]


def get_curriculum_loader(train_df, allowed_mags, nmt_cache, transform, batch_size):
    mask      = train_df["mag_label"].isin(allowed_mags)
    sub_df    = train_df[mask].reset_index(drop=True)
    sub_idx   = np.where(mask.values)[0]
    sub_cache = _SubNMTCache(nmt_cache, sub_idx)
    return _make_loader(BreakHisDataset(sub_df, sub_cache, transform),
                        batch_size, shuffle=True)


def get_curriculum_phase(epoch):
    for p in CURRICULUM:
        s, e = p["epochs"]
        if s <= epoch < e:
            return p["mags"]
    return {0, 1, 2, 3}


def tta_collate(batch):
    imgs, y_sub, y_bin, y_mag = zip(*batch)
    feats      = [_nmt_ext.extract(img) for img in imgs]
    nmt_toks   = torch.from_numpy(np.stack([f[0] for f in feats]))
    nmt_heats  = torch.from_numpy(np.stack([f[1] for f in feats]))
    nmt_sigs   = torch.from_numpy(np.stack([f[2] for f in feats]))
    nmt_stains = torch.from_numpy(np.stack([f[3] for f in feats]))
    return (list(imgs),
            torch.tensor(y_sub,  dtype=torch.long),
            torch.tensor(y_bin,  dtype=torch.long),
            torch.tensor(y_mag,  dtype=torch.long),
            nmt_toks, nmt_heats, nmt_sigs, nmt_stains)


# ═══════════════════════════════════════════════════════════════════════════
# MODEL COMPONENTS
# ═══════════════════════════════════════════════════════════════════════════

class _GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, scale):
        ctx.save_for_backward(torch.tensor(scale))
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        scale = ctx.saved_tensors[0].item()
        return -scale * grad, None


class GradientReversalLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("_iter", torch.zeros(1, dtype=torch.long))
    def forward(self, x):
        p     = float(self._iter.item()) / max(GRL_MAX_ITER, 1)
        scale = 2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0
        if self.training:
            self._iter += 1
        return _GRLFunction.apply(x, scale)


class StainDiscriminator(nn.Module):
    def __init__(self, in_dim: int = EMBED_DIM, out_dim: int = STAIN_FEAT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, out_dim),
        )
    def forward(self, x):
        return self.net(x)


class ExpertFFN(nn.Module):
    def __init__(self, dim: int = EMBED_DIM, mult: int = 4, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mult, dim),
        )
    def forward(self, x):
        return self.net(x)


class MagnificationConditionedMoE(nn.Module):
    def __init__(self,
                 dim:       int = EMBED_DIM,
                 n_experts: int = N_EXPERTS,
                 top_k:     int = MOE_TOP_K,
                 mag_dim:   int = MOE_MAG_DIM):
        super().__init__()
        self.n_experts = n_experts
        self.top_k     = top_k
        self.experts   = nn.ModuleList([ExpertFFN(dim) for _ in range(n_experts)])
        self.mag_embed = nn.Embedding(4, mag_dim)
        self.router    = nn.Linear(dim + mag_dim, n_experts, bias=False)

    def forward(self, x: torch.Tensor, mag: torch.Tensor):
        mag_emb   = self.mag_embed(mag)
        router_in = torch.cat([x, mag_emb], dim=-1)
        logits    = self.router(router_in)
        top_logits, top_idx = logits.topk(self.top_k, dim=-1)
        gate_weights = F.softmax(top_logits, dim=-1)
        all_out  = torch.stack([e(x) for e in self.experts], dim=1)
        idx_exp  = top_idx.unsqueeze(-1).expand(-1, -1, x.shape[-1])
        selected = all_out.gather(1, idx_exp)
        output   = (selected * gate_weights.unsqueeze(-1)).sum(1)
        output   = output + x
        full_probs = F.softmax(logits, dim=-1)
        one_hot    = F.one_hot(top_idx[:, 0], self.n_experts).float()
        f_e        = one_hot.mean(0)
        P_e        = full_probs.mean(0)
        load_loss  = self.n_experts * (f_e * P_e).sum()
        return output, load_loss


# ═══════════════════════════════════════════════════════════════════════════
# MAIN MODEL: CausalMorph-ViT (Tiny)
# ═══════════════════════════════════════════════════════════════════════════
class CausalMorphViT(nn.Module):
    def __init__(self,
                 num_subclasses: int   = 8,
                 dropout:        float = HEAD_DROPOUT,
                 proj_dim:       int   = PROJ_DIM,
                 n_nmt_tokens:   int   = N_NMT_TOKENS):
        super().__init__()
        self.vit = timm.create_model(
            "vit_tiny_patch16_224",
            pretrained  = True,
            num_classes = 0,
        )
        dim = self.vit.num_features   # 192

        self.nmt_prior_alpha = nn.Parameter(torch.tensor(0.30))
        self.n_nmt_tokens    = n_nmt_tokens
        self.nmt_proj = nn.Sequential(
            nn.Linear(NMT_FEAT_DIM, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
        )
        self.nmt_pos = nn.Parameter(torch.randn(1, n_nmt_tokens, dim) * 0.02)

        self.mc_moe = MagnificationConditionedMoE(dim)

        self.grl        = GradientReversalLayer()
        self.stain_disc = StainDiscriminator(dim, STAIN_FEAT_DIM)

        self.head_sub = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, num_subclasses))
        self.head_bin = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim, 2))

        self.projector = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(inplace=True),
            nn.Linear(dim, proj_dim),
        )

    @staticmethod
    def _pos_drop(vit, x):
        return vit.pos_drop(x) if hasattr(vit, "pos_drop") else x

    def forward(self,
                x:           torch.Tensor,
                mag:         torch.Tensor,
                nmt_tokens:  torch.Tensor,
                nmt_heatmap: torch.Tensor,
                return_proj: bool = False):

        B   = x.shape[0]
        vit = self.vit

        patches = vit.patch_embed(x)

        alpha   = self.nmt_prior_alpha.clamp(0.0, 2.0)
        prior_w = 1.0 + alpha * nmt_heatmap.unsqueeze(-1)
        patches = patches * prior_w

        cls_tok = vit.cls_token.expand(B, -1, -1)
        x_seq   = torch.cat([cls_tok, patches], dim=1)
        x_seq   = x_seq + vit.pos_embed
        x_seq   = self._pos_drop(vit, x_seq)

        n_tok = self.nmt_proj(nmt_tokens)
        n_tok = n_tok + self.nmt_pos
        x_seq = torch.cat([x_seq, n_tok], dim=1)

        for blk in vit.blocks:
            x_seq = blk(x_seq)
        x_seq = vit.norm(x_seq)
        feat  = x_seq[:, 0]

        feat_moe, moe_loss = self.mc_moe(feat, mag)

        out_sub = self.head_sub(feat_moe)
        out_bin = self.head_bin(feat_moe)

        stain_pred = self.stain_disc(self.grl(feat_moe))

        if return_proj:
            z = F.normalize(self.projector(feat_moe), dim=1)
            return out_sub, out_bin, z, feat_moe, moe_loss, stain_pred
        return out_sub, out_bin, moe_loss, stain_pred


# ── Prototype Bank ─────────────────────────────────────────────────────────
class PrototypeBank(nn.Module):
    def __init__(self, num_classes=8, feat_dim=EMBED_DIM, momentum=0.99):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("prototypes",  torch.zeros(num_classes, feat_dim))
        self.register_buffer("initialized", torch.zeros(num_classes, dtype=torch.bool))

    @torch.no_grad()
    def update(self, features, labels):
        for c in labels.unique():
            m  = labels == c
            fc = features[m].mean(0)
            if self.initialized[c]:
                self.prototypes[c] = self.momentum * self.prototypes[c] + (1-self.momentum) * fc
            else:
                self.prototypes[c] = fc; self.initialized[c] = True

    def prototype_loss(self, features, labels):
        if not self.initialized.any(): return features.new_zeros(1).squeeze()
        valid = self.initialized[labels]
        if not valid.any(): return features.new_zeros(1).squeeze()
        return F.mse_loss(features[valid], self.prototypes[labels[valid]].detach())


# ═══════════════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=FOCAL_GAMMA, label_smoothing=0.10):
        super().__init__()
        self.weight = weight; self.gamma = gamma; self.ls = label_smoothing

    def forward(self, logits, targets):
        log_p = F.log_softmax(logits, dim=1)
        pt    = log_p.exp().gather(1, targets.unsqueeze(1)).squeeze(1)
        fw    = (1 - pt) ** self.gamma
        nll_h = -log_p.gather(1, targets.unsqueeze(1)).squeeze(1)
        nll_s = -log_p.mean(dim=1)
        ce    = (1 - self.ls) * nll_h + self.ls * nll_s
        if self.weight is not None:
            w  = self.weight[targets]
            ce = ce * (w / self.weight.mean())
        return (fw * ce).mean()


class MorphologyTopologyContrastiveLoss(nn.Module):
    def __init__(self, temperature: float = TEMPERATURE):
        super().__init__()
        self.temp = temperature

    def forward(self, z, labels, nmt_sigs):
        B        = z.shape[0]
        z        = z.float()
        nmt_sigs = nmt_sigs.float()
        sig_norm  = F.normalize(nmt_sigs, dim=1)
        morph_sim = torch.mm(sig_norm, sig_norm.T)
        morph_w   = (morph_sim + 1.0) / 2.0
        sim = torch.mm(z, z.T) / self.temp
        eye = torch.eye(B, dtype=torch.bool, device=z.device)
        sim.masked_fill_(eye, -1e4)
        pos_mask = (labels.unsqueeze(1) == labels.unsqueeze(0)) & ~eye
        log_prob = F.log_softmax(sim, dim=1)
        weighted_pos = log_prob * pos_mask.float() * morph_w
        n_pos        = pos_mask.sum(1).float().clamp(min=1)
        loss         = -(weighted_pos.sum(1) / n_pos).mean()
        return loss


mtcl_loss_fn = MorphologyTopologyContrastiveLoss()


def _reduce_scalar_loss(t: torch.Tensor) -> torch.Tensor:
    if t.dim() == 0:
        return t
    return t.mean()


def compute_total_loss(out_sub, out_bin, z, feat, moe_loss, stain_pred,
                       y_sub, y_bin, nmt_sigs, stain_tgt,
                       focal_sub_fn, ce_bin_fn, proto_bank):
    l_cls   = focal_sub_fn(out_sub, y_sub) + 0.7 * ce_bin_fn(out_bin, y_bin)
    l_mtcl  = mtcl_loss_fn(z, y_sub, nmt_sigs)
    l_proto = proto_bank.prototype_loss(feat.detach(), y_sub)
    proto_bank.update(feat.detach(), y_sub)
    l_moe = _reduce_scalar_loss(moe_loss) if torch.is_tensor(moe_loss) \
            else torch.tensor(float(moe_loss), device=out_sub.device)
    l_stain = F.mse_loss(stain_pred, stain_tgt)
    total = (l_cls
           + LAMBDA_MTCL  * l_mtcl
           + LAMBDA_PROTO * l_proto
           + LAMBDA_MOE   * l_moe
           + LAMBDA_ASD   * l_stain)
    return (total,
            l_cls.item(), l_mtcl.item(),
            l_proto.item() if isinstance(l_proto, torch.Tensor) else 0.0,
            l_moe.item(), l_stain.item())


# ═══════════════════════════════════════════════════════════════════════════
# METRICS HELPERS
# ═══════════════════════════════════════════════════════════════════════════
def compute_gmean(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    return float(np.prod(per_class_recall) ** (1.0 / len(per_class_recall)))


def binary_metrics(y_true, y_pred, y_prob):
    acc  = (np.array(y_pred) == np.array(y_true)).mean()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    prec = precision_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    kap  = cohen_kappa_score(y_true, y_pred)
    gmn  = float(np.sqrt(sens * spec))
    auc_val = roc_auc_score(y_true, y_prob)
    return dict(acc=acc, auc=auc_val, f1=f1, sens=sens, spec=spec,
                prec=prec, mcc=mcc, kappa=kap, gmean=gmn)


def multiclass_metrics(y_true, y_pred, y_prob_mat, n_classes=8):
    acc  = (np.array(y_pred) == np.array(y_true)).mean()
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    kap  = cohen_kappa_score(y_true, y_pred)
    gmn  = compute_gmean(y_true, y_pred)
    cm   = confusion_matrix(y_true, y_pred)
    spec_list = []
    for c in range(n_classes):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        spec_list.append(tn / (tn + fp + 1e-8))
    spec = float(np.mean(spec_list))
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    try:
        auc_val = roc_auc_score(y_bin, y_prob_mat, multi_class="ovr", average="macro")
    except Exception:
        auc_val = float("nan")
    return dict(acc=acc, auc=auc_val, f1=f1, sens=rec, spec=spec,
                prec=prec, mcc=mcc, kappa=kap, gmean=gmn)


# ═══════════════════════════════════════════════════════════════════════════
# PLOTTING HELPERS
# ═══════════════════════════════════════════════════════════════════════════
def plot_confusion_matrix(cm, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(max(6, len(labels)), max(5, len(labels)-1)))
    cmap = plt.cm.Blues
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=np.arange(len(labels)), yticks=np.arange(len(labels)),
           xticklabels=labels, yticklabels=labels,
           title=title, ylabel="True label", xlabel="Predicted label")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor",
             fontsize=8 if len(labels) > 4 else 10)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], "d"),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else NAVY,
                    fontsize=7 if len(labels) > 4 else 9)
    fig.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 Saved: {save_path}")


def plot_roc_binary(y_true, y_prob, title, save_path, fold=None):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color=DEEP_BLUE, lw=2,
            label=f"ROC (AUC = {roc_auc:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.15, color=SKY_BLUE)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(loc="lower right")
    fig.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 Saved: {save_path}")


def plot_roc_multiclass(y_true, y_prob_mat, class_names, title, save_path):
    n_cls = len(class_names)
    y_bin = label_binarize(y_true, classes=list(range(n_cls)))
    cmap  = plt.cm.get_cmap("tab10", n_cls)
    fig, ax = plt.subplots(figsize=(9, 7))
    aucs = []
    for i, cname in enumerate(class_names):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob_mat[:, i])
            roc_auc = auc(fpr, tpr)
            aucs.append(roc_auc)
            ax.plot(fpr, tpr, color=cmap(i), lw=1.5,
                    label=f"{cname} (AUC={roc_auc:.3f})")
        except Exception:
            pass
    fpr_m, tpr_m, _ = roc_curve(y_bin.ravel(), y_prob_mat.ravel())
    macro_auc = np.mean(aucs) if aucs else 0.0
    ax.plot(fpr_m, tpr_m, color=NAVY, lw=2, linestyle="--",
            label=f"Macro AUC = {macro_auc:.4f}")
    ax.fill_between(fpr_m, tpr_m, alpha=0.05, color=SKY_BLUE)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=7, ncol=2)
    fig.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 Saved: {save_path}")


# ═══════════════════════════════════════════════════════════════════════════
# COMPUTATIONAL ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════
def get_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_flops(model):
    if not HAS_PTFLOPS:
        return "N/A (ptflops not installed)"
    try:
        dummy_x    = torch.zeros(1, 3, 224, 224).to(DEVICE)
        dummy_mag  = torch.zeros(1, dtype=torch.long).to(DEVICE)
        dummy_tok  = torch.zeros(1, N_NMT_TOKENS, NMT_FEAT_DIM).to(DEVICE)
        dummy_heat = torch.zeros(1, 196).to(DEVICE)

        class _Wrapped(nn.Module):
            def __init__(self, m): super().__init__(); self.m = m
            def forward(self, x):
                return self.m(dummy_x, dummy_mag, dummy_tok, dummy_heat)

        macs, params = get_model_complexity_info(
            _Wrapped(model), (3, 224, 224),
            as_strings=False, print_per_layer_stat=False)
        return f"{macs/1e9:.2f} GFLOPs"
    except Exception as e:
        return f"N/A ({e})"


# ═══════════════════════════════════════════════════════════════════════════
# ATTENTION ROLLOUT  (XAI)
# ═══════════════════════════════════════════════════════════════════════════
class AttentionRollout:
    def __init__(self, model: CausalMorphViT, discard_ratio=0.9):
        self.model        = model
        self.discard_ratio = discard_ratio
        self.hooks        = []
        self.attentions   = []

    def _hook(self, module, input, output):
        # ViT block returns x; we need the attention weights
        # timm ViT attention returns (x, attn_weights) when output_attentions=True
        # We monkey-patch the forward temporarily
        pass

    def get_rollout(self, x, mag, nmt_tokens, nmt_heatmap):
        self.model.eval()
        attentions = []

        def hook_fn(module, inp, outp):
            # Capture attention weights from the self-attention module
            # outp shape: (B, heads, seq, seq) or just x
            # We re-run attn to get weights
            pass

        # Manually run with attention collection
        vit    = self.model.vit
        B      = x.shape[0]
        patches = vit.patch_embed(x)
        alpha   = self.model.nmt_prior_alpha.clamp(0.0, 2.0)
        prior_w = 1.0 + alpha * nmt_heatmap.unsqueeze(-1)
        patches = patches * prior_w
        cls_tok = vit.cls_token.expand(B, -1, -1)
        x_seq   = torch.cat([cls_tok, patches], dim=1)
        x_seq   = x_seq + vit.pos_embed
        if hasattr(vit, "pos_drop"):
            x_seq = vit.pos_drop(x_seq)
        n_tok   = self.model.nmt_proj(nmt_tokens)
        n_tok   = n_tok + self.model.nmt_pos
        x_seq   = torch.cat([x_seq, n_tok], dim=1)

        # collect attention weights block by block
        for blk in vit.blocks:
            # run attention sub-module
            attn_module = blk.attn
            B2, N2, C2  = x_seq.shape
            qkv = attn_module.qkv(x_seq).reshape(
                B2, N2, 3, attn_module.num_heads,
                C2 // attn_module.num_heads).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            scale   = (C2 // attn_module.num_heads) ** -0.5
            attn_w  = (q @ k.transpose(-2, -1)) * scale
            attn_w  = attn_w.softmax(dim=-1)      # (B, heads, N, N)
            attentions.append(attn_w.detach().cpu())
            x_seq   = blk(x_seq)

        x_seq = vit.norm(x_seq)

        # rollout
        result = torch.eye(attentions[0].shape[-1])
        for a in attentions:
            a_avg  = a.mean(1).squeeze(0)  # (N, N)
            a_avg  = a_avg + torch.eye(a_avg.shape[0])
            a_avg  = a_avg / a_avg.sum(dim=-1, keepdim=True)
            result = torch.mm(a_avg, result)

        # only patch tokens (skip cls and nmt)
        n_patch = 196
        mask = result[0, 1:1+n_patch]  # (196,)
        mask = mask.numpy().reshape(14, 14)
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        return mask


def visualize_attention_rollout(model, samples_by_class, fold, prefix, out_dir, class_names):
    """
    samples_by_class: dict {class_id: (pil_img, mag_label)}
    """
    rollout = AttentionRollout(model)
    n_cls   = len(samples_by_class)
    cols    = 4
    rows    = (n_cls + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols * 2,
                             figsize=(cols * 2 * 2.5, rows * 3))
    axes = axes.flatten()

    idx = 0
    for cls_id, (pil_img, mag_val) in sorted(samples_by_class.items()):
        img_t  = val_tf(pil_img).unsqueeze(0).to(DEVICE)
        mag_t  = torch.tensor([mag_val], dtype=torch.long).to(DEVICE)
        feats  = _nmt_ext.extract(pil_img)
        tok_t  = torch.from_numpy(feats[0]).unsqueeze(0).to(DEVICE)
        heat_t = torch.from_numpy(feats[1]).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            mask = rollout.get_rollout(img_t, mag_t, tok_t, heat_t)

        img_np = np.array(pil_img.resize((224, 224)))
        mask_r = cv2.resize(mask, (224, 224))

        ax_orig = axes[idx * 2]
        ax_roll = axes[idx * 2 + 1]

        ax_orig.imshow(img_np)
        ax_orig.set_title(class_names.get(cls_id, str(cls_id)), fontsize=7, color=NAVY)
        ax_orig.axis("off")

        ax_roll.imshow(img_np)
        ax_roll.imshow(mask_r, cmap="Blues", alpha=0.55)
        ax_roll.set_title(f"Rollout", fontsize=7, color=NAVY)
        ax_roll.axis("off")

        idx += 1

    # hide remaining axes
    for ax in axes[idx * 2:]:
        ax.axis("off")

    fig.suptitle(f"Attention Rollout — Fold {fold} — {prefix}", fontsize=11, color=NAVY, y=1.01)
    fig.tight_layout()
    save_path = out_dir / f"xai_attention_rollout_fold{fold}_{prefix}.png"
    plt.savefig(str(save_path), dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 XAI saved: {save_path}")


# ═══════════════════════════════════════════════════════════════════════════
# ORIGINAL + CROP + ROI VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════
def visualize_original_crop_roi(df_test, fold, out_dir, n_per_class=1):
    """
    Show: original image | center crop | top-ROI patch  for each subclass.
    """
    cls_ids = sorted(df_test["subclass_label"].unique())
    n_cls   = len(cls_ids)
    cols    = 3
    rows    = n_cls

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 2.8))
    if rows == 1:
        axes = axes[np.newaxis, :]

    for r, cls_id in enumerate(cls_ids):
        sub = df_test[df_test["subclass_label"] == cls_id]
        # pick first image that exists
        row_data = None
        for _, rr in sub.iterrows():
            if Path(rr["path"]).exists():
                row_data = rr; break
        if row_data is None:
            for ax in axes[r]: ax.axis("off")
            continue

        path = row_data["path"]
        try:
            pil_orig = Image.open(path).convert("RGB")
        except Exception:
            for ax in axes[r]: ax.axis("off"); continue

        w, h = pil_orig.size
        # center crop
        cx, cy = w // 2, h // 2
        half   = min(w, h, IMG_SIZE) // 2
        pil_crop = pil_orig.crop((cx-half, cy-half, cx+half, cy+half)).resize((224, 224))

        # ROI: highest ESAP patch
        best_score, best_patch = -1, None
        for yy in range(0, h - IMG_SIZE + 1, STRIDE):
            for xx in range(0, w - IMG_SIZE + 1, STRIDE):
                patch = pil_orig.crop((xx, yy, xx+IMG_SIZE, yy+IMG_SIZE))
                s = esap_nmt_score(np.array(patch))
                if s > best_score:
                    best_score = s; best_patch = patch
        if best_patch is None:
            best_patch = pil_crop

        cname = SUBCLASS_NAMES.get(cls_id, str(cls_id))
        axes[r, 0].imshow(np.array(pil_orig.resize((224, 224))))
        axes[r, 0].set_title(f"{cname}\nOriginal", fontsize=7, color=NAVY)
        axes[r, 0].axis("off")

        axes[r, 1].imshow(np.array(pil_crop))
        axes[r, 1].set_title(f"Center Crop", fontsize=7, color=NAVY)
        axes[r, 1].axis("off")

        axes[r, 2].imshow(np.array(best_patch.resize((224, 224))))
        axes[r, 2].set_title(f"Top ROI Patch", fontsize=7, color=NAVY)
        axes[r, 2].axis("off")

    fig.suptitle(f"Original | Crop | ROI — Fold {fold}", fontsize=11, color=NAVY)
    fig.tight_layout()
    save_path = out_dir / f"orig_crop_roi_fold{fold}.png"
    plt.savefig(str(save_path), dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 Crop/ROI saved: {save_path}")


# ═══════════════════════════════════════════════════════════════════════════
# TRAIN / EVAL FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════
def _to(t): return t.to(DEVICE, non_blocking=True)


def train_one_epoch(loader, train_model, base_model, optimizer, scaler,
                    focal_sub_fn, ce_bin_fn, proto_bank):
    train_model.train()
    tot = cls_s = mtcl_s = proto_s = moe_s = stain_s = 0.0
    correct = 0
    for x, y_sub, y_bin, y_mag, n_tok, n_heat, n_sig, n_stain in loader:
        x, y_sub, y_bin = _to(x), _to(y_sub), _to(y_bin)
        y_mag, n_tok, n_heat = _to(y_mag), _to(n_tok), _to(n_heat)
        n_sig, n_stain = _to(n_sig), _to(n_stain)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda"):
            out_sub, out_bin, z, feat, moe_loss, stain_pred = train_model(
                x, y_mag, n_tok, n_heat, return_proj=True)
            loss, l_cls, l_mtcl, l_proto, l_moe, l_stain = compute_total_loss(
                out_sub, out_bin, z, feat, moe_loss, stain_pred,
                y_sub, y_bin, n_sig, n_stain,
                focal_sub_fn, ce_bin_fn, proto_bank)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        tot    += loss.item();  cls_s   += l_cls
        mtcl_s += l_mtcl;      proto_s += l_proto
        moe_s  += l_moe;       stain_s += l_stain
        correct += (out_sub.argmax(1) == y_sub).sum().item()
    n = len(loader)
    return (tot/n, cls_s/n, mtcl_s/n, proto_s/n, moe_s/n, stain_s/n,
            correct / len(loader.dataset))


@torch.no_grad()
def evaluate(loader, model, focal_sub_fn, ce_bin_fn):
    model.eval()
    tot_loss = correct = 0
    for x, y_sub, y_bin, y_mag, n_tok, n_heat, n_sig, n_stain in loader:
        x, y_sub, y_bin = _to(x), _to(y_sub), _to(y_bin)
        y_mag, n_tok, n_heat = _to(y_mag), _to(n_tok), _to(n_heat)
        with autocast(device_type="cuda"):
            out_sub, out_bin, _, _ = model(x, y_mag, n_tok, n_heat)
        tot_loss += (focal_sub_fn(out_sub, y_sub) + 0.7 * ce_bin_fn(out_bin, y_bin)).item()
        correct  += (out_sub.argmax(1) == y_sub).sum().item()
    return tot_loss / len(loader), correct / len(loader.dataset)


@torch.no_grad()
def collect_predictions(loader, model, n_classes=8):
    """Returns sub_preds, sub_probs, sub_labels, bin_preds, bin_probs, bin_labels"""
    model.eval()
    sub_preds, sub_probs, sub_labels = [], [], []
    bin_preds, bin_probs, bin_labels = [], [], []
    for x, y_sub, y_bin, y_mag, n_tok, n_heat, _, _ in loader:
        x = _to(x); y_sub = _to(y_sub); y_bin = _to(y_bin)
        y_mag = _to(y_mag); n_tok = _to(n_tok); n_heat = _to(n_heat)
        with autocast(device_type="cuda"):
            out_sub, out_bin, _, _ = model(x, y_mag, n_tok, n_heat)
        sub_preds.extend(out_sub.argmax(1).cpu().numpy())
        sub_probs.extend(F.softmax(out_sub, dim=1).cpu().numpy())
        sub_labels.extend(y_sub.cpu().numpy())
        bin_preds.extend(out_bin.argmax(1).cpu().numpy())
        bin_probs.extend(F.softmax(out_bin, dim=1)[:, 1].cpu().numpy())
        bin_labels.extend(y_bin.cpu().numpy())
    return (np.array(sub_preds), np.array(sub_probs),
            np.array(sub_labels), np.array(bin_preds),
            np.array(bin_probs), np.array(bin_labels))


@torch.no_grad()
def collect_predictions_tta(pil_dataset, model, nmt_cache_for_tta, n_classes=8):
    """TTA predictions using 10 augmentation views."""
    model.eval()
    tta_loader = DataLoader(pil_dataset, batch_size=INFER_BATCH,
                            shuffle=False, num_workers=NUM_WORKERS,
                            collate_fn=tta_collate, persistent_workers=False)
    sub_preds, bin_preds   = [], []
    sub_probs, bin_probs   = [], []
    sub_labels, bin_labels = [], []
    for pil_imgs, y_sub, y_bin, y_mag, n_tok, n_heat, n_sig, n_stain in tta_loader:
        y_sub = _to(y_sub); y_bin = _to(y_bin); y_mag = _to(y_mag)
        n_tok = _to(n_tok); n_heat = _to(n_heat)
        B = y_sub.size(0)
        avg_sub = torch.zeros(B, n_classes, device=DEVICE)
        avg_bin = torch.zeros(B, 2, device=DEVICE)
        for tf in tta_transforms:
            bt = torch.stack([tf(img) for img in pil_imgs]).to(DEVICE)
            with autocast(device_type="cuda"):
                out_sub, out_bin, _, _ = model(bt, y_mag, n_tok, n_heat)
            avg_sub += out_sub; avg_bin += out_bin
        avg_sub /= len(tta_transforms); avg_bin /= len(tta_transforms)
        sub_preds.extend(avg_sub.argmax(1).cpu().numpy())
        sub_probs.extend(F.softmax(avg_sub, dim=1).cpu().numpy())
        sub_labels.extend(y_sub.cpu().numpy())
        bin_preds.extend(avg_bin.argmax(1).cpu().numpy())
        bin_probs.extend(F.softmax(avg_bin, dim=1)[:, 1].cpu().numpy())
        bin_labels.extend(y_bin.cpu().numpy())
    return (np.array(sub_preds), np.array(sub_probs),
            np.array(sub_labels), np.array(bin_preds),
            np.array(bin_probs), np.array(bin_labels))


# ═══════════════════════════════════════════════════════════════════════════
# 5-FOLD CROSS VALIDATION MAIN LOOP
# ═══════════════════════════════════════════════════════════════════════════
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# All fold results
all_fold_results = []

# Cross-fold aggregate predictions for final averaged plots
agg_bin_labels_tta, agg_bin_probs_tta   = [], []
agg_sub_labels_tta, agg_sub_probs_tta   = [], []
agg_bin_labels,     agg_bin_probs       = [], []
agg_sub_labels,     agg_sub_probs       = [], []

# Computational stats
comp_params    = None
comp_flops     = None
comp_epoch_times_all = []
comp_img_times_all   = []

print("\n" + "═"*60)
print("  5-FOLD CROSS VALIDATION — CausalMorph-ViT-Tiny")
print("═"*60)

X_all = df_all["path"].values
y_all = df_all["subclass_label"].values

for fold_idx, (trainval_idx, test_idx) in enumerate(
        skf.split(X_all, y_all)):

    fold = fold_idx + 1
    print(f"\n{'─'*60}")
    print(f"  FOLD {fold} / {N_FOLDS}")
    print(f"{'─'*60}")

    # ── Split: 80% train+val, 20% test ────────────────────────────────────
    df_trainval = df_all.iloc[trainval_idx].reset_index(drop=True)
    df_test_raw = df_all.iloc[test_idx].reset_index(drop=True)

    # Further split train+val → 90% train, 10% val (image level)
    skf_inner = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED+fold)
    tv_X = df_trainval["path"].values
    tv_y = df_trainval["subclass_label"].values
    for inner_train_idx, inner_val_idx in skf_inner.split(tv_X, tv_y):
        break  # take first split → ~90/10

    df_train_raw = df_trainval.iloc[inner_train_idx].reset_index(drop=True)
    df_val_raw   = df_trainval.iloc[inner_val_idx].reset_index(drop=True)

    print(f"  Train: {len(df_train_raw)}  |  Val: {len(df_val_raw)}  |  Test: {len(df_test_raw)}")

    # ── ROI augmentation on train ─────────────────────────────────────────
    fold_tag  = f"fold{fold}"
    df_train  = apply_roi_to_df(df_train_raw, fold_tag)

    # ── NMT Caches ────────────────────────────────────────────────────────
    print(f"\n📐 Building NMT caches for fold {fold} …")
    train_nmt = NMTCache(df_train,   _nmt_ext, tag=f"train_{fold_tag}")
    val_nmt   = NMTCache(df_val_raw, _nmt_ext, tag=f"val_{fold_tag}")
    test_nmt  = NMTCache(df_test_raw,_nmt_ext, tag=f"test_{fold_tag}")

    # ── Class weights (from augmented train) ──────────────────────────────
    _w = compute_class_weight("balanced", classes=np.arange(8),
                              y=df_train["subclass_label"].values)
    class_weights = torch.tensor(_w, dtype=torch.float).to(DEVICE)
    focal_sub_fn  = FocalLoss(weight=class_weights)
    ce_bin_fn     = nn.CrossEntropyLoss(label_smoothing=0.10)

    # ── Build model ────────────────────────────────────────────────────────
    base_model = CausalMorphViT().to(DEVICE)
    if NUM_GPUS > 1:
        train_model = nn.DataParallel(base_model, device_ids=list(range(NUM_GPUS)))
        print(f"\n  ⚡ DataParallel activated across {NUM_GPUS} GPUs")
    else:
        train_model = base_model

    if comp_params is None:
        comp_params = get_trainable_params(base_model)
        print(f"\n  🔢 Trainable Parameters : {comp_params:,}")

    if comp_flops is None:
        comp_flops = get_flops(base_model)
        print(f"  🔢 FLOPs               : {comp_flops}")

    proto_bank = PrototypeBank().to(DEVICE)

    # ── Optimizer + Scheduler ─────────────────────────────────────────────
    optimizer    = torch.optim.AdamW(base_model.parameters(), lr=3e-4,
                                     weight_decay=WEIGHT_DECAY)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
                       optimizer, start_factor=0.1, end_factor=1.0,
                       total_iters=WARMUP_EPOCHS)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                       optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
    scheduler    = torch.optim.lr_scheduler.SequentialLR(
                       optimizer, [warmup_sched, cosine_sched],
                       milestones=[WARMUP_EPOCHS])
    scaler       = GradScaler(device="cuda")

    # ── Val / Test loaders ─────────────────────────────────────────────────
    val_loader  = _make_loader(BreakHisDataset(df_val_raw,  val_nmt,  val_tf),
                               INFER_BATCH, False)
    test_loader = _make_loader(BreakHisDataset(df_test_raw, test_nmt, val_tf),
                               INFER_BATCH, False)

    ckpt_path = OUT_DIR / f"best_causal_morph_vit_tiny_fold{fold}.pth"

    # ── Training loop ─────────────────────────────────────────────────────
    best_acc = 0.0
    epoch_times = []
    fold_history = []

    for epoch in range(EPOCHS):
        active_mags  = get_curriculum_phase(epoch)
        train_loader = get_curriculum_loader(
            df_train, active_mags, train_nmt, train_tf, TRAIN_BATCH)

        phase = ("Phase-1(400X)" if epoch < 15 else
                 "Phase-2(400/200/100X)" if epoch < 30 else "Phase-3(ALL)")

        t0 = time.time()
        (tr_loss, l_cls, l_mtcl, l_proto,
         l_moe, l_stain, tr_acc) = train_one_epoch(
            train_loader, train_model, base_model, optimizer, scaler,
            focal_sub_fn, ce_bin_fn, proto_bank)
        val_loss, val_acc = evaluate(val_loader, base_model,
                                     focal_sub_fn, ce_bin_fn)
        epoch_t = time.time() - t0
        epoch_times.append(epoch_t)

        scheduler.step()
        lr_now = optimizer.param_groups[0]["lr"]

        fold_history.append(dict(
            epoch=epoch+1, phase=phase, lr=lr_now,
            tr_loss=tr_loss, val_loss=val_loss,
            tr_acc=tr_acc, val_acc=val_acc, epoch_time=epoch_t))

        # Print only selected epochs
        if (epoch + 1) in {1, 10, 20, 30, 40, 45}:
            print(f"    Epoch {epoch+1:02d}/{EPOCHS}"
                  f"| Tr  Loss={tr_loss:.4f} Acc={tr_acc:.4f}"
                  f" | Val Loss={val_loss:.4f} Acc={val_acc:.4f}"
                  f" | ⏱ {epoch_t:.1f}s")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(base_model.state_dict(), str(ckpt_path))

        del train_loader

    mean_epoch_time = np.mean(epoch_times)
    n_train_imgs    = len(df_train)
    mean_per_img_ms = (mean_epoch_time / max(n_train_imgs, 1)) * 1000
    comp_epoch_times_all.append(mean_epoch_time)
    comp_img_times_all.append(mean_per_img_ms)

    print(f"\n  ✅ Best Val Acc : {best_acc:.4f}")
    print(f"  ⏱  Mean Epoch  : {mean_epoch_time:.2f} s  |"
          f"  Mean Per-Image : {mean_per_img_ms:.4f} ms")

    # ── Load best checkpoint ──────────────────────────────────────────────
    base_model.load_state_dict(torch.load(str(ckpt_path), map_location=DEVICE))
    base_model.eval()
    final_model = base_model

    # ── Evaluate without TTA ──────────────────────────────────────────────
    (sp_nt, sp_prob_nt, sl_nt,
     bp_nt, bp_prob_nt, bl_nt) = collect_predictions(
        test_loader, final_model)

    bin_m_nt  = binary_metrics(bl_nt, bp_nt, bp_prob_nt)
    mult_m_nt = multiclass_metrics(sl_nt, sp_nt, sp_prob_nt)

    # ── Evaluate with TTA ─────────────────────────────────────────────────
    tta_test_ds = BreakHisTTADataset(df_test_raw)
    (sp_tta, sp_prob_tta, sl_tta,
     bp_tta, bp_prob_tta, bl_tta) = collect_predictions_tta(
        tta_test_ds, final_model, test_nmt)

    bin_m_tta  = binary_metrics(bl_tta, bp_tta, bp_prob_tta)
    mult_m_tta = multiclass_metrics(sl_tta, sp_tta, sp_prob_tta)

    # aggregate for cross-fold
    agg_bin_labels.extend(bl_nt.tolist());       agg_bin_probs.extend(bp_prob_nt.tolist())
    agg_sub_labels.extend(sl_nt.tolist());       agg_sub_probs.extend(sp_prob_nt.tolist())
    agg_bin_labels_tta.extend(bl_tta.tolist());  agg_bin_probs_tta.extend(bp_prob_tta.tolist())
    agg_sub_labels_tta.extend(sl_tta.tolist());  agg_sub_probs_tta.extend(sp_prob_tta.tolist())

    # ── Print fold results ────────────────────────────────────────────────
    print(f"\n  Fold {fold} Results:")
    print(f"  Binary(without TTA)     | "
          f"Acc={bin_m_nt['acc']:.4f}  AUC={bin_m_nt['auc']:.4f}  "
          f"F1={bin_m_nt['f1']:.4f}  Sens={bin_m_nt['sens']:.4f}  "
          f"Spec={bin_m_nt['spec']:.4f}  MCC={bin_m_nt['mcc']:.4f}  "
          f"Kappa={bin_m_nt['kappa']:.4f}  G-mean={bin_m_nt['gmean']:.4f}")
    print(f"  Binary(with TTA)        | "
          f"Acc={bin_m_tta['acc']:.4f}  AUC={bin_m_tta['auc']:.4f}  "
          f"F1={bin_m_tta['f1']:.4f}  Sens={bin_m_tta['sens']:.4f}  "
          f"Spec={bin_m_tta['spec']:.4f}  MCC={bin_m_tta['mcc']:.4f}  "
          f"Kappa={bin_m_tta['kappa']:.4f}  G-mean={bin_m_tta['gmean']:.4f}")
    print(f"  Multiclass(without TTA) | "
          f"Acc={mult_m_nt['acc']:.4f}  AUC={mult_m_nt['auc']:.4f}  "
          f"F1={mult_m_nt['f1']:.4f}  Sens={mult_m_nt['sens']:.4f}  "
          f"Spec={mult_m_nt['spec']:.4f}  MCC={mult_m_nt['mcc']:.4f}  "
          f"Kappa={mult_m_nt['kappa']:.4f}  G-mean={mult_m_nt['gmean']:.4f}")
    print(f"  Multiclass(with TTA)    | "
          f"Acc={mult_m_tta['acc']:.4f}  AUC={mult_m_tta['auc']:.4f}  "
          f"F1={mult_m_tta['f1']:.4f}  Sens={mult_m_tta['sens']:.4f}  "
          f"Spec={mult_m_tta['spec']:.4f}  MCC={mult_m_tta['mcc']:.4f}  "
          f"Kappa={mult_m_tta['kappa']:.4f}  G-mean={mult_m_tta['gmean']:.4f}")

    # ── Save fold confusion matrices ──────────────────────────────────────
    # Binary no-TTA
    cm_bin = confusion_matrix(bl_nt, bp_nt, labels=[0,1])
    plot_confusion_matrix(cm_bin, BINARY_NAMES,
        f"Binary CM — Fold {fold} (no TTA)",
        str(OUT_DIR / f"cm_binary_fold{fold}_noTTA.png"))

    # Binary TTA
    cm_bin_tta = confusion_matrix(bl_tta, bp_tta, labels=[0,1])
    plot_confusion_matrix(cm_bin_tta, BINARY_NAMES,
        f"Binary CM — Fold {fold} (TTA)",
        str(OUT_DIR / f"cm_binary_fold{fold}_TTA.png"))

    # Multiclass no-TTA
    cm_sub = confusion_matrix(sl_nt, sp_nt, labels=list(range(8)))
    plot_confusion_matrix(cm_sub,
        [SUBCLASS_NAMES[i] for i in range(8)],
        f"Multiclass CM — Fold {fold} (no TTA)",
        str(OUT_DIR / f"cm_multiclass_fold{fold}_noTTA.png"))

    # Multiclass TTA
    cm_sub_tta = confusion_matrix(sl_tta, sp_tta, labels=list(range(8)))
    plot_confusion_matrix(cm_sub_tta,
        [SUBCLASS_NAMES[i] for i in range(8)],
        f"Multiclass CM — Fold {fold} (TTA)",
        str(OUT_DIR / f"cm_multiclass_fold{fold}_TTA.png"))

    # ── ROC curves ────────────────────────────────────────────────────────
    plot_roc_binary(bl_nt, bp_prob_nt,
        f"Binary ROC — Fold {fold} (no TTA)",
        str(OUT_DIR / f"roc_binary_fold{fold}_noTTA.png"))
    plot_roc_binary(bl_tta, bp_prob_tta,
        f"Binary ROC — Fold {fold} (TTA)",
        str(OUT_DIR / f"roc_binary_fold{fold}_TTA.png"))
    plot_roc_multiclass(sl_nt, sp_prob_nt,
        [SUBCLASS_NAMES[i] for i in range(8)],
        f"Multiclass ROC — Fold {fold} (no TTA)",
        str(OUT_DIR / f"roc_multiclass_fold{fold}_noTTA.png"))
    plot_roc_multiclass(sl_tta, sp_prob_tta,
        [SUBCLASS_NAMES[i] for i in range(8)],
        f"Multiclass ROC — Fold {fold} (TTA)",
        str(OUT_DIR / f"roc_multiclass_fold{fold}_TTA.png"))

    # ── XAI: Attention Rollout ────────────────────────────────────────────
    # Collect one sample per subclass and per binary class
    samples_subclass = {}
    samples_binary   = {}
    for _, row in df_test_raw.iterrows():
        cls_id = int(row["subclass_label"])
        bin_id = int(row["binary_label"])
        if len(samples_subclass) < 8 and cls_id not in samples_subclass:
            if Path(row["path"]).exists():
                try:
                    pil = Image.open(row["path"]).convert("RGB")
                    samples_subclass[cls_id] = (pil, int(row["mag_label"]))
                except Exception:
                    pass
        if len(samples_binary) < 2 and bin_id not in samples_binary:
            if Path(row["path"]).exists():
                try:
                    pil = Image.open(row["path"]).convert("RGB")
                    samples_binary[bin_id] = (pil, int(row["mag_label"]))
                except Exception:
                    pass
        if len(samples_subclass) == 8 and len(samples_binary) == 2:
            break

    visualize_attention_rollout(
        final_model, samples_subclass, fold, "subclass_8types",
        OUT_DIR, SUBCLASS_NAMES)
    visualize_attention_rollout(
        final_model, samples_binary, fold, "binary_2classes",
        OUT_DIR, {0: "benign", 1: "malignant"})

    # ── Original | Crop | ROI visualization ──────────────────────────────
    visualize_original_crop_roi(df_test_raw, fold, OUT_DIR)

    # ── Training history plot ─────────────────────────────────────────────
    hist_df = pd.DataFrame(fold_history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist_df["epoch"], hist_df["tr_loss"],
                 color=DEEP_BLUE, label="Train Loss")
    axes[0].plot(hist_df["epoch"], hist_df["val_loss"],
                 color=SKY_BLUE, label="Val Loss", linestyle="--")
    axes[0].set_title(f"Loss — Fold {fold}", color=NAVY)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[1].plot(hist_df["epoch"], hist_df["tr_acc"],
                 color=DEEP_BLUE, label="Train Acc")
    axes[1].plot(hist_df["epoch"], hist_df["val_acc"],
                 color=SKY_BLUE, label="Val Acc", linestyle="--")
    axes[1].set_title(f"Accuracy — Fold {fold}", color=NAVY)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    fig.tight_layout()
    plt.savefig(str(OUT_DIR / f"history_fold{fold}.png"), dpi=130, bbox_inches="tight")
    plt.close(fig)

    # Store fold results
    all_fold_results.append(dict(
        fold=fold,
        best_val_acc=best_acc,
        mean_epoch_time=mean_epoch_time,
        mean_per_img_ms=mean_per_img_ms,
        bin_nt=bin_m_nt,
        bin_tta=bin_m_tta,
        mult_nt=mult_m_nt,
        mult_tta=mult_m_tta,
    ))

    # Save per-fold history CSV
    hist_df.to_csv(str(OUT_DIR / f"history_fold{fold}.csv"), index=False)

    # Cleanup to free VRAM
    del base_model, train_model, final_model
    del train_nmt, val_nmt, test_nmt
    del optimizer, scaler, proto_bank
    torch.cuda.empty_cache()
    print(f"\n{'─'*60}")


# ═══════════════════════════════════════════════════════════════════════════
# CROSS-FOLD AGGREGATE RESULTS
# ═══════════════════════════════════════════════════════════════════════════
def _mean_std(vals):
    return float(np.mean(vals)), float(np.std(vals))

metrics_keys = ["acc", "auc", "f1", "sens", "spec", "prec", "mcc", "kappa", "gmean"]

print("\n" + "═"*70)
print("  CROSS-FOLD AGGREGATE RESULTS (mean ± std)")
print("═"*70)

for mode_name, mode_key in [("Binary (no TTA)", "bin_nt"),
                             ("Binary (TTA)",    "bin_tta"),
                             ("Multiclass (no TTA)", "mult_nt"),
                             ("Multiclass (TTA)",    "mult_tta")]:
    print(f"\n  {mode_name}:")
    for k in metrics_keys:
        vals = [r[mode_key][k] for r in all_fold_results]
        mu, sd = _mean_std(vals)
        print(f"    {k:10s}: {mu:.4f} ± {sd:.4f}")

# ── Computational summary ──────────────────────────────────────────────────
print(f"\n  Computational Analysis:")
print(f"    Trainable Parameters   : {comp_params:,}")
print(f"    FLOPs                  : {comp_flops}")
print(f"    Mean Train Time/Epoch  : {np.mean(comp_epoch_times_all):.2f} ± "
      f"{np.std(comp_epoch_times_all):.2f} s")
print(f"    Mean Train Time/Image  : {np.mean(comp_img_times_all):.4f} ± "
      f"{np.std(comp_img_times_all):.4f} ms")

# ── Aggregate cross-fold confusion matrices ────────────────────────────────
# Binary TTA
cm_agg_bin = confusion_matrix(agg_bin_labels_tta, np.array(agg_bin_probs_tta) > 0.5,
                               labels=[0,1])
plot_confusion_matrix(cm_agg_bin, BINARY_NAMES,
    "Aggregated Binary CM (All Folds — TTA)",
    str(OUT_DIR / "cm_binary_allfolds_TTA.png"))

# Multiclass TTA
agg_sub_preds = np.argmax(np.array(agg_sub_probs_tta), axis=1)
cm_agg_sub = confusion_matrix(agg_sub_labels_tta, agg_sub_preds, labels=list(range(8)))
plot_confusion_matrix(cm_agg_sub,
    [SUBCLASS_NAMES[i] for i in range(8)],
    "Aggregated Multiclass CM (All Folds — TTA)",
    str(OUT_DIR / "cm_multiclass_allfolds_TTA.png"))

# ── Aggregate ROC curves ───────────────────────────────────────────────────
plot_roc_binary(agg_bin_labels_tta, agg_bin_probs_tta,
    "Aggregated Binary ROC (All Folds — TTA)",
    str(OUT_DIR / "roc_binary_allfolds_TTA.png"))

plot_roc_multiclass(agg_sub_labels_tta, np.array(agg_sub_probs_tta),
    [SUBCLASS_NAMES[i] for i in range(8)],
    "Aggregated Multiclass ROC (All Folds — TTA)",
    str(OUT_DIR / "roc_multiclass_allfolds_TTA.png"))

# ── Summary table figure ───────────────────────────────────────────────────
def make_summary_table(all_fold_results, mode_key, mode_name, save_path):
    keys = metrics_keys
    fold_vals = {k: [r[mode_key][k] for r in all_fold_results] for k in keys}
    mu_row = {k: np.mean(fold_vals[k]) for k in keys}
    sd_row = {k: np.std(fold_vals[k])  for k in keys}

    col_labels = [f"Fold {r['fold']}" for r in all_fold_results] + ["Mean", "Std"]
    row_labels  = [k.upper() for k in keys]

    cell_data = []
    for k in keys:
        row = [f"{v:.4f}" for v in fold_vals[k]]
        row += [f"{mu_row[k]:.4f}", f"{sd_row[k]:.4f}"]
        cell_data.append(row)

    fig, ax = plt.subplots(figsize=(len(col_labels) * 1.4, len(keys) * 0.55 + 1.5))
    ax.axis("off")
    tbl = ax.table(cellText=cell_data, rowLabels=row_labels, colLabels=col_labels,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    tbl.scale(1.0, 1.6)

    # Header colors
    for j in range(len(col_labels)):
        tbl[0, j].set_facecolor(SKY_BLUE)
        tbl[0, j].set_text_props(color=NAVY, fontweight="bold")
    for i in range(len(keys)):
        tbl[i+1, -1].set_facecolor(LIGHT_GRAY)

    ax.set_title(f"Summary Table — {mode_name}", pad=12, color=NAVY, fontsize=10)
    fig.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 Table saved: {save_path}")


make_summary_table(all_fold_results, "bin_nt",   "Binary (no TTA)",
    str(OUT_DIR / "summary_binary_noTTA.png"))
make_summary_table(all_fold_results, "bin_tta",  "Binary (TTA)",
    str(OUT_DIR / "summary_binary_TTA.png"))
make_summary_table(all_fold_results, "mult_nt",  "Multiclass (no TTA)",
    str(OUT_DIR / "summary_multiclass_noTTA.png"))
make_summary_table(all_fold_results, "mult_tta", "Multiclass (TTA)",
    str(OUT_DIR / "summary_multiclass_TTA.png"))

# ── Computational summary bar chart ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
folds_x = [r["fold"] for r in all_fold_results]

axes[0].bar(folds_x, [r["mean_epoch_time"] for r in all_fold_results],
            color=SKY_BLUE, edgecolor=NAVY)
axes[0].set_title("Mean Epoch Time (s)", color=NAVY)
axes[0].set_xlabel("Fold"); axes[0].set_ylabel("Seconds")

axes[1].bar(folds_x, [r["mean_per_img_ms"] for r in all_fold_results],
            color=DEEP_BLUE, edgecolor=NAVY)
axes[1].set_title("Mean Time per Image (ms)", color=NAVY)
axes[1].set_xlabel("Fold"); axes[1].set_ylabel("ms")

axes[2].bar(folds_x, [r["best_val_acc"] for r in all_fold_results],
            color=SKY_BLUE, edgecolor=NAVY)
axes[2].set_title("Best Val Acc per Fold", color=NAVY)
axes[2].set_xlabel("Fold"); axes[2].set_ylabel("Accuracy")
axes[2].set_ylim([0.85, 1.0])

params_str = f"Params: {comp_params:,} | FLOPs: {comp_flops}"
fig.suptitle(f"Computational Analysis — CausalMorph-ViT-Tiny\n{params_str}",
             fontsize=10, color=NAVY)
fig.tight_layout()
plt.savefig(str(OUT_DIR / "computational_analysis.png"), dpi=130, bbox_inches="tight")
plt.close(fig)
print(f"   💾 Computational analysis saved.")

# ── Save full results CSV ─────────────────────────────────────────────────
rows = []
for r in all_fold_results:
    base = {"fold": r["fold"],
            "best_val_acc": r["best_val_acc"],
            "mean_epoch_time_s": r["mean_epoch_time"],
            "mean_per_img_ms": r["mean_per_img_ms"]}
    for mode_key, mode_name in [("bin_nt","bin_noTTA"),("bin_tta","bin_TTA"),
                                 ("mult_nt","mult_noTTA"),("mult_tta","mult_TTA")]:
        for k in metrics_keys:
            base[f"{mode_name}_{k}"] = r[mode_key][k]
    rows.append(base)
pd.DataFrame(rows).to_csv(str(OUT_DIR / "5fold_results.csv"), index=False)
print(f"   💾 Results CSV saved: 5fold_results.csv")

# ═══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY PRINT
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"📊 CausalMorph-ViT-Tiny — 5-FOLD FINAL SUMMARY")
print(f"{'='*70}")
print(f"  Backbone       : ViT-Tiny/16 (~5.7M params, embed={EMBED_DIM})")
print(f"  [NMT]          : {N_NMT_TOKENS}×12-dim nucleus tokens + density prior")
print(f"  [ASD]          : Macenko 6-dim GRL adversarial stain loss (λ={LAMBDA_ASD})")
print(f"  [MC-MoE]       : {N_EXPERTS} experts, top-{MOE_TOP_K}, mag-conditioned")
print(f"  [MTCL]         : Nuclear-weighted SupCon (λ={LAMBDA_MTCL})")
print(f"  Trainable Params : {comp_params:,}")
print(f"  FLOPs            : {comp_flops}")
print(f"  Mean Epoch Time  : {np.mean(comp_epoch_times_all):.2f} s")
print(f"  Mean Per-Image   : {np.mean(comp_img_times_all):.4f} ms")
print(f"{'─'*70}")

for mode_name, mode_key in [("Binary      (no TTA)", "bin_nt"),
                             ("Binary      (TTA)   ", "bin_tta"),
                             ("Multiclass  (no TTA)", "mult_nt"),
                             ("Multiclass  (TTA)   ", "mult_tta")]:
    acc_vals = [r[mode_key]["acc"] for r in all_fold_results]
    auc_vals = [r[mode_key]["auc"] for r in all_fold_results]
    f1_vals  = [r[mode_key]["f1"]  for r in all_fold_results]
    print(f"  {mode_name} | "
          f"Acc={np.mean(acc_vals):.4f}±{np.std(acc_vals):.4f}  "
          f"AUC={np.mean(auc_vals):.4f}±{np.std(auc_vals):.4f}  "
          f"F1={np.mean(f1_vals):.4f}±{np.std(f1_vals):.4f}")

print(f"{'='*70}")
print(f"\n💾 Output files in /kaggle/working/:")
saved_files = [
    "5fold_results.csv",
    "summary_binary_noTTA.png", "summary_binary_TTA.png",
    "summary_multiclass_noTTA.png", "summary_multiclass_TTA.png",
    "cm_binary_allfolds_TTA.png", "cm_multiclass_allfolds_TTA.png",
    "roc_binary_allfolds_TTA.png", "roc_multiclass_allfolds_TTA.png",
    "computational_analysis.png",
]
for i in range(1, N_FOLDS + 1):
    saved_files += [
        f"best_causal_morph_vit_tiny_fold{i}.pth",
        f"history_fold{i}.csv", f"history_fold{i}.png",
        f"cm_binary_fold{i}_noTTA.png", f"cm_binary_fold{i}_TTA.png",
        f"cm_multiclass_fold{i}_noTTA.png", f"cm_multiclass_fold{i}_TTA.png",
        f"roc_binary_fold{i}_noTTA.png", f"roc_binary_fold{i}_TTA.png",
        f"roc_multiclass_fold{i}_noTTA.png", f"roc_multiclass_fold{i}_TTA.png",
        f"xai_attention_rollout_fold{i}_subclass_8types.png",
        f"xai_attention_rollout_fold{i}_binary_2classes.png",
        f"orig_crop_roi_fold{i}.png",
    ]
for f in saved_files:
    print(f"   📄 /kaggle/working/{f}")

# 5 fold result zip (png, csv)

In [ ]:
import os
import shutil
import zipfile

# 1. Create the results folder
os.makedirs('/kaggle/working/results', exist_ok=True)

# 2. Copy all .png and .csv files into results/
import glob

for f in glob.glob('/kaggle/working/*.png') + glob.glob('/kaggle/working/*.csv'):
    shutil.copy(f, '/kaggle/working/results/')

# 3. Zip the results folder
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/', 'results')

print("Done! Download results.zip from /kaggle/working/results.zip")

# 5 fold results zip(pth)

In [ ]:
import os
import shutil
import glob

# 1. Create the results_pth folder
os.makedirs('/kaggle/working/results_pth', exist_ok=True)

# 2. Copy all .pth files into results_pth/
for f in glob.glob('/kaggle/working/*.pth'):
    shutil.copy(f, '/kaggle/working/results_pth/')

# 3. Zip the results_pth folder
shutil.make_archive('/kaggle/working/results_pth', 'zip', '/kaggle/working/', 'results_pth')

print("Done! Download results_pth.zip from /kaggle/working/results_pth.zip")

In [ ]:
import zipfile
import os

output_dir = '/kaggle/working'
zip_filename = os.path.join(output_dir, 'xai_v1.zip')

files = [
    'gradcam_subclass_fold1_improved.png', 'gradcam_binary_fold1_improved.png',
    'attention_rollout_subclass_fold1_improved.png', 'attention_rollout_binary_fold1_improved.png',
    'integrated_gradients_subclass_fold1_improved.png', 'integrated_gradients_binary_fold1_improved.png',
    'xai_comparison_subclass_fold1_improved.png', 'xai_comparison_binary_fold1_improved.png',
    'gradcam_subclass_fold2_improved.png', 'gradcam_binary_fold2_improved.png',
    'attention_rollout_subclass_fold2_improved.png', 'attention_rollout_binary_fold2_improved.png',
    'integrated_gradients_subclass_fold2_improved.png', 'integrated_gradients_binary_fold2_improved.png',
    'xai_comparison_subclass_fold2_improved.png', 'xai_comparison_binary_fold2_improved.png',
    'gradcam_subclass_fold3_improved.png', 'gradcam_binary_fold3_improved.png',
    'attention_rollout_subclass_fold3_improved.png', 'attention_rollout_binary_fold3_improved.png',
    'integrated_gradients_subclass_fold3_improved.png', 'integrated_gradients_binary_fold3_improved.png',
    'xai_comparison_subclass_fold3_improved.png', 'xai_comparison_binary_fold3_improved.png',
    'gradcam_subclass_fold4_improved.png', 'gradcam_binary_fold4_improved.png',
    'attention_rollout_subclass_fold4_improved.png', 'attention_rollout_binary_fold4_improved.png',
    'integrated_gradients_subclass_fold4_improved.png', 'integrated_gradients_binary_fold4_improved.png',
    'xai_comparison_subclass_fold4_improved.png', 'xai_comparison_binary_fold4_improved.png',
    'gradcam_subclass_fold5_improved.png', 'gradcam_binary_fold5_improved.png',
    'attention_rollout_subclass_fold5_improved.png', 'attention_rollout_binary_fold5_improved.png',
    'integrated_gradients_subclass_fold5_improved.png', 'integrated_gradients_binary_fold5_improved.png',
    'xai_comparison_subclass_fold5_improved.png', 'xai_comparison_binary_fold5_improved.png',
]

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in files:
        fpath = os.path.join(output_dir, fname)
        if os.path.exists(fpath):
            zf.write(fpath, arcname=os.path.join('xai_v1', fname))
            print(f'Added: {fname}')
        else:
            print(f'MISSING: {fname}')

print(f'\nDone! Saved to: {zip_filename}')

# XAI - final - v3

In [ ]:
#!/usr/bin/env python3
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  XAI Runner — CausalMorph-ViT  (v3_final)                              ║
# ║  Run this as a NEW CELL at the very end of your Kaggle notebook         ║
# ║  Methods : GradCAM · Attention Rollout · Integrated Gradients           ║
# ║  Output  : paper-quality 300 DPI PNGs → zipped as xai_v3_final.zip     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# ── REQUIREMENTS ──────────────────────────────────────────────────────────
#   DEVICE, OUT_DIR, SUBCLASS_NAMES, N_FOLDS, SEED
#   val_tf, _nmt_ext
#   CausalMorphViT  (the model class)
#   df_all          (the full CSV dataframe)
#   StratifiedKFold (already imported)
# ─────────────────────────────────────────────────────────────────────────

import torch
import torch.nn.functional as F
import numpy as np
import cv2
import warnings
import zipfile
from PIL import Image
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════════════════════════════════
# CONFIG  ←─ Edit here only
# ═══════════════════════════════════════════════════════════════════════════
RUN_FOLDS       = [1]                       # ← single fold; change to e.g. [3]
CKPT_DIR        = Path("/kaggle/working")
XAI_OUT         = Path("/kaggle/working")
IG_STEPS        = 50
ROLLOUT_DISCARD = 0.90
PAPER_DPI       = 300

# Decision-region highlight
HIGHLIGHT_COLOR     = "#FFD700"
HIGHLIGHT_THRESHOLD = 0.72      # pixels above this get a contour ring
HIGHLIGHT_LW        = 1.8       # contour line width


# ═══════════════════════════════════════════════════════════════════════════
# COLOUR MAPS
# ═══════════════════════════════════════════════════════════════════════════
CMAP_GRADCAM = LinearSegmentedColormap.from_list(
    "gcam", ["#0d0221","#1a0a4c","#0d3b91","#1565c0",
             "#00acc1","#00e5ff","#76ff03","#ffea00","#ff6d00","#dd2222"])

CMAP_ROLLOUT = LinearSegmentedColormap.from_list(
    "roll", ["#000000","#001f5b","#003c8f","#0070c0",
             "#00b0ff","#40c4ff","#80d8ff","#ffffff"])

CMAP_IG = LinearSegmentedColormap.from_list(
    "ig",   ["#4a148c","#7b1fa2","#ab47bc","#ce93d8",
             "#f8bbd0","#ffffff","#b9f6ca","#00c853","#007c34"])

OVERLAY_ALPHA = 0.58

TITLE_FONT = dict(fontsize=9,  fontweight="bold",
                  color="#0d1b2a", fontfamily="DejaVu Sans")
LABEL_FONT = dict(fontsize=7.0, color="#1a1a2e",
                  fontfamily="DejaVu Sans")

plt.rcParams.update({
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": False,
})


# ═══════════════════════════════════════════════════════════════════════════
# PLOT HELPERS
# ═══════════════════════════════════════════════════════════════════════════
def _add_colorbar(fig, ax, cmap, label="", vmin=0, vmax=1):
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.03)
    sm  = plt.cm.ScalarMappable(cmap=cmap,
                                norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label(label, fontsize=5.5, color="#1a1a2e")
    cb.ax.tick_params(labelsize=5, colors="#1a1a2e")
    cb.outline.set_edgecolor("#cccccc")


def _overlay(img_np, heatmap, cmap, alpha=OVERLAY_ALPHA):
    h_rgb = (cmap(heatmap)[:, :, :3] * 255).astype(np.uint8)
    return (img_np.astype(np.float32) * (1 - alpha) +
            h_rgb.astype(np.float32) * alpha).clip(0, 255).astype(np.uint8)


def _border(ax, color="#ff6d00", lw=2.5):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(lw)


def _decision_highlight(ax, hmap, threshold=HIGHLIGHT_THRESHOLD):
    """
    Draw a contour ring around the highest-activation decision region.
    Solid primary boundary + subtle dotted inner ring for paper visibility.
    """
    try:
        region_mask = np.where(hmap >= threshold, 1.0, np.nan)
        ax.imshow(region_mask,
                  cmap=LinearSegmentedColormap.from_list(
                      "dr", ["#FFD70000", "#FFD70040"]),
                  vmin=0, vmax=1, alpha=0.35, interpolation="bilinear")
        ax.contour(hmap, levels=[threshold],
                   colors=[HIGHLIGHT_COLOR],
                   linewidths=HIGHLIGHT_LW,
                   linestyles="solid", alpha=0.95)
        ax.contour(hmap, levels=[threshold + 0.10],
                   colors=["#FFFACD"],
                   linewidths=0.8,
                   linestyles="dotted", alpha=0.80)
    except Exception:
        pass


def _is_malignant(cid, head):
    return cid in set(range(4, 8)) if head == "sub" else cid == 1


def _prep_tensors(pil_img, mag_val):
    x      = val_tf(pil_img).unsqueeze(0).to(DEVICE)
    mag    = torch.tensor([mag_val], dtype=torch.long).to(DEVICE)
    feats  = _nmt_ext.extract(pil_img)
    tok    = torch.from_numpy(feats[0]).unsqueeze(0).to(DEVICE)
    heat   = torch.from_numpy(feats[1]).unsqueeze(0).to(DEVICE)
    img224 = np.array(pil_img.resize((224, 224)))
    return x, mag, tok, heat, img224


def _collect_samples(df, class_col, class_ids):
    samples = {}
    for _, row in df.iterrows():
        cid = int(row[class_col])
        if cid in class_ids and cid not in samples:
            p = Path(row["path"])
            if p.exists():
                try:
                    pil = Image.open(p).convert("RGB")
                    samples[cid] = (pil, int(row["mag_label"]), str(p))
                except Exception:
                    pass
        if len(samples) == len(class_ids):
            break
    return samples


def _rebuild_test_df(fold):
    X_all = df_all["path"].values
    y_all = df_all["subclass_label"].values
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold_idx, (_, test_idx) in enumerate(skf.split(X_all, y_all)):
        if fold_idx + 1 == fold:
            return df_all.iloc[test_idx].reset_index(drop=True)
    raise ValueError(f"Fold {fold} not found")


# ═══════════════════════════════════════════════════════════════════════════
# XAI METHODS
# ═══════════════════════════════════════════════════════════════════════════

# ── 1. Grad-CAM ──────────────────────────────────────────────────────────
class GradCAMViT:
    """
    FIX — Black output was caused by positive/negative gradient cancellation
    in global-average pooling step. Solution: use ReLU-gated gradient weights
    (keep only positive-contributing channels) + absolute-value fallback.
    """
    def __init__(self, model):
        self.model = model
        self._acts = self._grads = None
        self._hf = self._hb = None

    def _register(self):
        def fwd(module, inp, out):
            self._acts = out.detach()
        def bwd(module, g_in, g_out):
            self._grads = g_out[0].detach()
        self._hf = self.model.vit.blocks[-1].register_forward_hook(fwd)
        self._hb = self.model.vit.blocks[-1].register_full_backward_hook(bwd)

    def _remove(self):
        if self._hf: self._hf.remove()
        if self._hb: self._hb.remove()

    def __call__(self, x, mag, tok, heat, target_class=None, head="sub"):
        self.model.eval()
        self._register()
        with torch.enable_grad():
            out_sub, out_bin, _, _ = self.model(
                x, mag, tok, heat, return_proj=False)
            logits = out_sub if head == "sub" else out_bin
            if target_class is None:
                target_class = logits.argmax(1).item()
            self.model.zero_grad()
            logits[0, target_class].backward()
        self._remove()

        acts    = self._acts[0]
        grads   = self._grads[0]
        p_acts  = acts[1:197]
        p_grads = grads[1:197]

        pos_weights = F.relu(p_grads).mean(dim=0)
        if pos_weights.max() < 1e-7:
            pos_weights = p_grads.abs().mean(dim=0)

        cam = (p_acts * pos_weights).sum(dim=-1)
        cam = F.relu(cam)

        if cam.max() < 1e-7:
            cam = p_acts.norm(dim=-1)

        cam = cam.reshape(14, 14).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (224, 224), interpolation=cv2.INTER_CUBIC)
        return np.clip(cam, 0, 1), target_class


# ── 2. Attention Rollout ─────────────────────────────────────────────────
class AttentionRolloutImproved:
    def __init__(self, model, discard_ratio=0.90):
        self.model         = model
        self.discard_ratio = discard_ratio

    @torch.no_grad()
    def __call__(self, x, mag, tok, heat):
        self.model.eval()
        vit = self.model.vit
        B   = x.shape[0]

        patches = vit.patch_embed(x)
        alpha   = self.model.nmt_prior_alpha.clamp(0.0, 2.0)
        patches = patches * (1.0 + alpha * heat.unsqueeze(-1))
        cls_tok = vit.cls_token.expand(B, -1, -1)
        x_seq   = torch.cat([cls_tok, patches], dim=1) + vit.pos_embed
        if hasattr(vit, "pos_drop"):
            x_seq = vit.pos_drop(x_seq)
        n_tok = self.model.nmt_proj(tok) + self.model.nmt_pos
        x_seq = torch.cat([x_seq, n_tok], dim=1)

        attentions = []
        for blk in vit.blocks:
            am        = blk.attn
            B2, N2, C2 = x_seq.shape
            qkv = am.qkv(x_seq).reshape(
                B2, N2, 3, am.num_heads,
                C2 // am.num_heads).permute(2, 0, 3, 1, 4)
            q, k, _ = qkv.unbind(0)
            aw = (q @ k.transpose(-2, -1) *
                  (C2 // am.num_heads) ** -0.5).softmax(-1)
            attentions.append(aw.mean(1).squeeze(0).cpu())
            x_seq = blk(x_seq)

        result = torch.eye(attentions[0].shape[-1])
        for a in attentions:
            flat   = a.flatten()
            thresh = flat.kthvalue(
                int(self.discard_ratio * flat.numel())).values.item()
            a      = torch.where(a < thresh, torch.zeros_like(a), a)
            a      = (a + torch.eye(a.shape[0]))
            a      = a / (a.sum(-1, keepdim=True) + 1e-8)
            result = torch.mm(a, result)

        mask = result[0, 1:197].numpy().reshape(14, 14)
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        return cv2.resize(mask, (224, 224), interpolation=cv2.INTER_CUBIC)


# ── 3. Integrated Gradients ──────────────────────────────────────────────
class IntegratedGradientsViT:
    def __init__(self, model, steps=50):
        self.model = model
        self.steps = steps

    def __call__(self, x, mag, tok, heat, target_class=None, head="sub"):
        self.model.eval()
        baseline   = torch.zeros_like(x)
        alphas     = torch.linspace(0, 1, self.steps, device=x.device)
        grads_list = []
        for a in alphas:
            inp = (baseline + a * (x - baseline)).requires_grad_(True)
            out_sub, out_bin, _, _ = self.model(
                inp, mag, tok, heat, return_proj=False)
            logits = out_sub if head == "sub" else out_bin
            if target_class is None:
                target_class = logits.argmax(1).item()
            self.model.zero_grad()
            logits[0, target_class].backward()
            grads_list.append(inp.grad.detach().clone())

        ig     = ((x - baseline) * torch.stack(grads_list).mean(0))[0]
        ig_map = ig.abs().sum(0).cpu().numpy()
        ig_map = (ig_map - ig_map.min()) / (ig_map.max() - ig_map.min() + 1e-8)
        return ig_map, target_class


# ── Smart dispatcher ─────────────────────────────────────────────────────
def _dispatch(method, x, mag, tok, heat, target_class, head):
    if isinstance(method, AttentionRolloutImproved):
        hmap = method(x, mag, tok, heat)
        return hmap, target_class
    else:
        return method(x, mag, tok, heat,
                      target_class=target_class, head=head)


# ═══════════════════════════════════════════════════════════════════════════
# FIGURE BUILDERS
# ═══════════════════════════════════════════════════════════════════════════

# ── A. Single-method subtype grid: 4 rows × 2 subtypes, 3 panels each ───
def _subtype_grid_figure(samples_dict, name_dict, method, cmap,
                         out_path, head):
    """
    Layout: 4 outer-rows × 2 subtypes per row.
    Each subtype cell: Original | Heatmap | Overlay  (3 panels).
    """
    ids = sorted(samples_dict.keys())

    N_SUBS_PER_ROW = 2
    N_PANEL_COLS   = 3
    N_ROWS         = 4

    fig_w = N_SUBS_PER_ROW * N_PANEL_COLS * 2.0 + 0.6
    fig_h = N_ROWS * 2.2 + 1.2

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=PAPER_DPI)
    fig.patch.set_facecolor("white")

    outer = gridspec.GridSpec(
        N_ROWS, N_SUBS_PER_ROW, figure=fig,
        hspace=0.45, wspace=0.10,
        top=0.96, bottom=0.07, left=0.02, right=0.99)

    col_labels = ["Original", "Heatmap", "Overlay"]

    for i, cid in enumerate(ids):
        row_idx = i // N_SUBS_PER_ROW
        col_idx = i %  N_SUBS_PER_ROW

        pil_img, mag_val, _ = samples_dict[cid]
        x, mag, tok, heat, img224 = _prep_tensors(pil_img, mag_val)

        with torch.enable_grad():
            hmap, _ = _dispatch(method, x, mag, tok, heat,
                                 target_class=cid, head=head)

        inner = gridspec.GridSpecFromSubplotSpec(
            1, N_PANEL_COLS,
            subplot_spec=outer[row_idx, col_idx],
            wspace=0.04)

        cls_name = name_dict.get(cid, str(cid))
        border_c = "#dd2222" if _is_malignant(cid, head) else "#1565c0"

        for p in range(N_PANEL_COLS):
            ax = fig.add_subplot(inner[p])
            ax.axis("off")

            if p == 0:
                ax.imshow(img224)
                _border(ax, border_c, lw=2.2)
                ax.set_title(cls_name, **TITLE_FONT, pad=2.5)
            elif p == 1:
                ax.imshow(hmap, cmap=cmap, vmin=0, vmax=1)
                _add_colorbar(fig, ax, cmap)
            else:
                ax.imshow(_overlay(img224, hmap, cmap))
                _decision_highlight(ax, hmap)

            if row_idx == 0 and col_idx == 0:
                ax.text(0.5, 1.16, col_labels[p],
                        transform=ax.transAxes,
                        ha="center", **LABEL_FONT, fontweight="bold")

    b_p = mpatches.Patch(color="#1565c0", label="Benign")
    m_p = mpatches.Patch(color="#dd2222", label="Malignant")
    h_p = mpatches.Patch(facecolor=HIGHLIGHT_COLOR, edgecolor="#B8860B",
                         label=f"Decision region (>{int(HIGHLIGHT_THRESHOLD*100)}%)")
    fig.legend(handles=[b_p, m_p, h_p], loc="lower center", ncol=3,
               fontsize=6.5, frameon=True, framealpha=0.88,
               bbox_to_anchor=(0.5, 0.0))

    plt.savefig(str(out_path), dpi=PAPER_DPI,
                bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"   💾 {Path(out_path).name}")


# ── B. Single-method binary figure: 2 rows × 4 columns ──────────────────
def _binary_figure(samples_dict, name_dict, method, cmap,
                   out_path, head):
    """
    2 classes (Benign / Malignant).
    Columns: Original | Heatmap | Overlay | Activation Histogram
    """
    ids = sorted(samples_dict.keys())
    n   = len(ids)

    fig = plt.figure(figsize=(4 * 2.2 * 1.05, n * 2.5 + 0.9), dpi=PAPER_DPI)
    fig.patch.set_facecolor("white")
    outer = gridspec.GridSpec(n, 1, figure=fig, hspace=0.38,
                              top=0.97, bottom=0.04, left=0.04, right=0.97)
    col_titles = ["Original", "Heatmap", "Overlay"]

    for row_i, cid in enumerate(ids):
        pil_img, mag_val, _ = samples_dict[cid]
        x, mag, tok, heat, img224 = _prep_tensors(pil_img, mag_val)

        with torch.enable_grad():
            hmap, _ = _dispatch(method, x, mag, tok, heat,
                                 target_class=cid, head=head)

        inner    = gridspec.GridSpecFromSubplotSpec(
            1, 4, subplot_spec=outer[row_i],
            wspace=0.08, width_ratios=[1, 1, 1, 0.9])
        cls_name = name_dict.get(cid, str(cid))
        border_c = "#dd2222" if _is_malignant(cid, head) else "#1565c0"

        # Original
        ax0 = fig.add_subplot(inner[0])
        ax0.imshow(img224); ax0.axis("off"); _border(ax0, border_c)
        ax0.set_title(cls_name, **TITLE_FONT, pad=3)
        if row_i == 0:
            ax0.text(0.5, 1.14, col_titles[0], transform=ax0.transAxes,
                     ha="center", **LABEL_FONT, fontweight="bold")

        # Heatmap
        ax1 = fig.add_subplot(inner[1])
        ax1.imshow(hmap, cmap=cmap, vmin=0, vmax=1); ax1.axis("off")
        _add_colorbar(fig, ax1, cmap)
        if row_i == 0:
            ax1.text(0.5, 1.14, col_titles[1], transform=ax1.transAxes,
                     ha="center", **LABEL_FONT, fontweight="bold")

        # Overlay + decision region highlight
        ax2 = fig.add_subplot(inner[2])
        ax2.imshow(_overlay(img224, hmap, cmap)); ax2.axis("off")
        _decision_highlight(ax2, hmap)
        if row_i == 0:
            ax2.text(0.5, 1.14, col_titles[2], transform=ax2.transAxes,
                     ha="center", **LABEL_FONT, fontweight="bold")

        # Activation histogram
        ax3 = fig.add_subplot(inner[3])
        vals = hmap.flatten()
        ax3.hist(vals, bins=40, color="#1565c0", alpha=0.75, edgecolor="none")
        ax3.axvline(vals.mean(), color="#ff6d00", lw=1.5,
                    label=f"μ={vals.mean():.2f}")
        ax3.axvline(HIGHLIGHT_THRESHOLD, color=HIGHLIGHT_COLOR, lw=1.4,
                    linestyle="--",
                    label=f"thr={HIGHLIGHT_THRESHOLD:.2f}")
        ax3.set_xlim([0, 1]); ax3.set_xticks([0, 0.5, 1])
        ax3.tick_params(labelsize=5.5); ax3.set_yticks([])
        ax3.legend(fontsize=5, loc="upper right", frameon=True, framealpha=0.7)
        ax3.spines["bottom"].set_visible(True)
        ax3.spines["bottom"].set_color("#cccccc")

    b_p = mpatches.Patch(color="#1565c0", label="Benign")
    m_p = mpatches.Patch(color="#dd2222", label="Malignant")
    h_p = mpatches.Patch(facecolor=HIGHLIGHT_COLOR, edgecolor="#B8860B",
                         label=f"Decision region (>{int(HIGHLIGHT_THRESHOLD*100)}%)")
    fig.legend(handles=[b_p, m_p, h_p], loc="lower center", ncol=3,
               fontsize=7, frameon=True, framealpha=0.85,
               bbox_to_anchor=(0.5, 0.0))

    plt.savefig(str(out_path), dpi=PAPER_DPI,
                bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"   💾 {Path(out_path).name}")


# ── C. 3-Method Comparison ───────────────────────────────────────────────
def _comparison_figure(samples_dict, name_dict, gcam, rollout, ig_fn,
                       out_path, head, use_grid=False):
    """
    3-method comparison — every method gets identical 3 columns:
        Original | Heatmap | Overlay
    = 9 visible panels per sample + 2 thin spacers between method groups.

    Column header (first sample only):
        ┌────────────────────┬────────────────────┬─────────────────────┐
        │      GradCAM       │  Attention Rollout  │ Integrated Gradients│
        │ Orig | Heat | Over │ Orig | Heat | Over  │ Orig | Heat | Over  │
        └────────────────────┴────────────────────┴─────────────────────┘

    For subtypes (use_grid=True) : tiled 4×2 outer grid.
    For binary  (use_grid=False) : single-column N-row layout.
    """
    ids      = sorted(samples_dict.keys())
    n        = len(ids)
    cmaps    = [CMAP_GRADCAM, CMAP_ROLLOUT, CMAP_IG]
    m_labels = ["GradCAM", "Attention Rollout", "Integrated Gradients"]
    col_titles = ["Original", "Heatmap", "Overlay"]

    N_METHODS       = 3
    COLS_PER_METHOD = 3
    N_SPACERS       = N_METHODS - 1   # 2
    TOTAL_SLOTS     = N_METHODS * COLS_PER_METHOD + N_SPACERS  # 11

    # width_ratios: [1,1,1, 0.08, 1,1,1, 0.08, 1,1,1]
    W_RATIOS = []
    for m in range(N_METHODS):
        W_RATIOS += [1, 1, 1]
        if m < N_METHODS - 1:
            W_RATIOS.append(0.08)

    if use_grid:
        N_SUBS_PER_ROW = 2
        N_ROWS         = (n + N_SUBS_PER_ROW - 1) // N_SUBS_PER_ROW
        cell_w = COLS_PER_METHOD * N_METHODS * 0.85 + 0.5
        fig_w  = cell_w * N_SUBS_PER_ROW + 0.4
        fig_h  = N_ROWS * 2.4 + 1.2
        fig = plt.figure(figsize=(fig_w, fig_h), dpi=PAPER_DPI)
        fig.patch.set_facecolor("white")
        outer = gridspec.GridSpec(
            N_ROWS, N_SUBS_PER_ROW, figure=fig,
            hspace=0.48, wspace=0.06,
            top=0.94, bottom=0.07, left=0.01, right=0.99)
    else:
        cell_w = COLS_PER_METHOD * N_METHODS * 1.05 + 0.5
        fig_w  = cell_w
        fig_h  = n * 2.8 + 1.2
        fig = plt.figure(figsize=(fig_w, fig_h), dpi=PAPER_DPI)
        fig.patch.set_facecolor("white")
        outer = gridspec.GridSpec(
            n, 1, figure=fig, hspace=0.40,
            top=0.95, bottom=0.06, left=0.02, right=0.98)

    for i, cid in enumerate(ids):
        pil_img, mag_val, _ = samples_dict[cid]
        x, mag, tok, heat, img224 = _prep_tensors(pil_img, mag_val)

        with torch.enable_grad():
            gc_map, _ = _dispatch(gcam,  x, mag, tok, heat, cid, head)
            ig_map, _ = _dispatch(ig_fn, x, mag, tok, heat, cid, head)
        ro_map = rollout(x, mag, tok, heat)

        hmaps = [gc_map, ro_map, ig_map]

        if use_grid:
            row_idx = i // N_SUBS_PER_ROW
            col_idx = i %  N_SUBS_PER_ROW
            inner = gridspec.GridSpecFromSubplotSpec(
                1, TOTAL_SLOTS,
                subplot_spec=outer[row_idx, col_idx],
                wspace=0.04, width_ratios=W_RATIOS)
        else:
            inner = gridspec.GridSpecFromSubplotSpec(
                1, TOTAL_SLOTS,
                subplot_spec=outer[i],
                wspace=0.05, width_ratios=W_RATIOS)

        cls_name = name_dict.get(cid, str(cid))
        border_c = "#dd2222" if _is_malignant(cid, head) else "#1565c0"
        is_first = (i == 0)

        for m_idx in range(N_METHODS):
            slot0 = m_idx * (COLS_PER_METHOD + 1)
            hmap  = hmaps[m_idx]
            cmap  = cmaps[m_idx]

            for p in range(COLS_PER_METHOD):
                slot = slot0 + p
                ax   = fig.add_subplot(inner[slot])
                ax.axis("off")

                if p == 0:                       # Original
                    ax.imshow(img224)
                    _border(ax, border_c, lw=2.0)
                    if m_idx == 0:
                        ax.set_title(cls_name, **TITLE_FONT, pad=2.5)

                elif p == 1:                     # Heatmap
                    ax.imshow(hmap, cmap=cmap, vmin=0, vmax=1)
                    _add_colorbar(fig, ax, cmap)

                else:                            # Overlay + decision highlight
                    ax.imshow(_overlay(img224, hmap, cmap))
                    _decision_highlight(ax, hmap)

                # Column headers for first sample only
                if is_first:
                    ax.text(0.5, 1.20, col_titles[p],
                            transform=ax.transAxes,
                            ha="center", **LABEL_FONT)
                    if p == 1:
                        ax.text(0.5, 1.38, m_labels[m_idx],
                                transform=ax.transAxes,
                                ha="center", fontsize=7.5,
                                fontweight="bold", color="#0d1b2a",
                                fontfamily="DejaVu Sans")

            # Invisible spacer between method groups
            if m_idx < N_METHODS - 1:
                ax_sp = fig.add_subplot(inner[slot0 + COLS_PER_METHOD])
                ax_sp.axis("off")

    b_p = mpatches.Patch(color="#1565c0", label="Benign")
    m_p = mpatches.Patch(color="#dd2222", label="Malignant")
    h_p = mpatches.Patch(facecolor=HIGHLIGHT_COLOR, edgecolor="#B8860B",
                         label=f"Decision region (>{int(HIGHLIGHT_THRESHOLD*100)}%)")
    fig.legend(handles=[b_p, m_p, h_p], loc="lower center", ncol=3,
               fontsize=6.5, frameon=True, framealpha=0.85,
               bbox_to_anchor=(0.5, 0.002))

    plt.savefig(str(out_path), dpi=PAPER_DPI,
                bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"   💾 {Path(out_path).name}")


# ═══════════════════════════════════════════════════════════════════════════
# MAIN XAI RUNNER  (per fold)
# ═══════════════════════════════════════════════════════════════════════════
def run_xai_for_fold(fold: int):
    ft        = f"fold{fold}"
    ckpt_path = CKPT_DIR / f"best_causal_morph_vit_tiny_{ft}.pth"

    print(f"\n{'═'*60}")
    print(f"  XAI — {ft.upper()}  |  {ckpt_path.name}")
    print(f"{'═'*60}")

    if not ckpt_path.exists():
        print(f"  ⚠️  Checkpoint not found — skipping fold {fold}")
        return

    model = CausalMorphViT().to(DEVICE)
    model.load_state_dict(torch.load(str(ckpt_path), map_location=DEVICE))
    model.eval()
    print(f"  ✅ Model loaded")

    df_test = _rebuild_test_df(fold)
    print(f"  📋 Test set : {len(df_test)} images")

    subs      = _collect_samples(df_test, "subclass_label", set(range(8)))
    bins      = _collect_samples(df_test, "binary_label",   {0, 1})
    bin_names = {0: "Benign", 1: "Malignant"}

    for miss in (set(range(8)) - set(subs.keys())):
        print(f"  ⚠️  Missing subclass sample: {miss}")
    for miss in ({0, 1} - set(bins.keys())):
        print(f"  ⚠️  Missing binary  sample : {miss}")

    gcam    = GradCAMViT(model)
    rollout = AttentionRolloutImproved(model, discard_ratio=ROLLOUT_DISCARD)
    ig_fn   = IntegratedGradientsViT(model, steps=IG_STEPS)

    print(f"\n  🎨 Generating plots …")

    # ── GradCAM ─────────────────────────────────────────────────────────
    _subtype_grid_figure(
        subs, SUBCLASS_NAMES, gcam, CMAP_GRADCAM,
        XAI_OUT / f"gradcam_subclass_{ft}_xai_v3_final.png",
        head="sub")

    _binary_figure(
        bins, bin_names, gcam, CMAP_GRADCAM,
        XAI_OUT / f"gradcam_binary_{ft}_xai_v3_final.png",
        head="bin")

    # ── Attention Rollout ────────────────────────────────────────────────
    _subtype_grid_figure(
        subs, SUBCLASS_NAMES, rollout, CMAP_ROLLOUT,
        XAI_OUT / f"attention_rollout_subclass_{ft}_xai_v3_final.png",
        head="sub")

    _binary_figure(
        bins, bin_names, rollout, CMAP_ROLLOUT,
        XAI_OUT / f"attention_rollout_binary_{ft}_xai_v3_final.png",
        head="bin")

    # ── Integrated Gradients ─────────────────────────────────────────────
    _subtype_grid_figure(
        subs, SUBCLASS_NAMES, ig_fn, CMAP_IG,
        XAI_OUT / f"integrated_gradients_subclass_{ft}_xai_v3_final.png",
        head="sub")

    _binary_figure(
        bins, bin_names, ig_fn, CMAP_IG,
        XAI_OUT / f"integrated_gradients_binary_{ft}_xai_v3_final.png",
        head="bin")

    # ── 3-Method Comparison ──────────────────────────────────────────────
    _comparison_figure(
        subs, SUBCLASS_NAMES, gcam, rollout, ig_fn,
        XAI_OUT / f"xai_comparison_subclass_{ft}_xai_v3_final.png",
        head="sub",
        use_grid=True)

    _comparison_figure(
        bins, bin_names, gcam, rollout, ig_fn,
        XAI_OUT / f"xai_comparison_binary_{ft}_xai_v3_final.png",
        head="bin",
        use_grid=False)

    del model, gcam, rollout, ig_fn
    torch.cuda.empty_cache()
    print(f"\n  ✅ {ft} done — 8 PNGs saved")


# ═══════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'█'*60}")
print(f"  XAI Runner — CausalMorph-ViT (v3_final)")
print(f"  Fold(s) to process : {RUN_FOLDS}")
print(f"  Output directory   : {XAI_OUT}")
print(f"{'█'*60}")

for fold in RUN_FOLDS:
    run_xai_for_fold(fold)

# ═══════════════════════════════════════════════════════════════════════════
# ZIP ALL OUTPUTS → xai_v3_final.zip
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print(f"  📦 Packing all *_xai_v3_final.png → xai_v3_final.zip …")

zip_path  = XAI_OUT / "xai_v3_final.zip"
png_files = sorted(XAI_OUT.glob("*_xai_v3_final.png"))

with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:
    for png in png_files:
        zf.write(str(png), arcname=str(Path("xai_v3_final") / png.name))
        print(f"   ✔ {png.name}")

print(f"\n  ✅ Zip created : {zip_path}")
print(f"  Total files    : {len(png_files)}")
print(f"  Zip size       : {zip_path.stat().st_size / 1024:.1f} KB")
print(f"{'█'*60}")
print(f"  🎉  ALL XAI COMPLETE")
print(f"{'█'*60}")

# External Dataset(BUSI)

In [ ]:
#!/usr/bin/env python3
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  BUSI External Dataset Evaluation — CausalMorph-ViT                    ║
# ║  Tests: Zero-Shot · 1-Shot · 5-Shot · Fine-Tuned                       ║
# ║  Classes: benign · malignant · normal  (3-class)                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# ── REQUIREMENTS ──────────────────────────────────────────────────────────
#   Run AFTER the main CausalMorph-ViT training notebook.
#   The following must be already defined in the notebook scope:
#     DEVICE, SUBCLASS_NAMES, N_NMT_TOKENS, NMT_FEAT_DIM, NMT_SIG_DIM,
#     STAIN_FEAT_DIM, EMBED_DIM, HEAD_DROPOUT, PROJ_DIM, N_EXPERTS,
#     MOE_TOP_K, MOE_MAG_DIM, GRL_MAX_ITER, FOCAL_GAMMA, TEMPERATURE,
#     LAMBDA_MTCL, LAMBDA_PROTO, LAMBDA_MOE, LAMBDA_ASD, WARMUP_EPOCHS,
#     val_tf, train_tf, _nmt_ext, CausalMorphViT, NuclearMorphometryExtractor,
#     FocalLoss, MorphologyTopologyContrastiveLoss, PrototypeBank,
#     compute_total_loss, mtcl_loss_fn
# ─────────────────────────────────────────────────────────────────────────

import os, random, warnings, time, copy
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════
# BUSI CONFIG
# ═══════════════════════════════════════════════════════════════════════════
BUSI_ROOT  = Path("/kaggle/input/datasets/sabahesaraki/"
                  "breast-ultrasound-images-dataset/Dataset_BUSI_with_GT")
CKPT_PATH  = Path("/kaggle/working/best_causal_morph_vit_tiny_fold3.pth")
BUSI_OUT   = Path("/kaggle/working/busi_eval")
BUSI_OUT.mkdir(parents=True, exist_ok=True)

# BUSI has 3 classes
BUSI_CLASSES    = ["benign", "malignant", "normal"]
BUSI_CLASS2ID   = {c: i for i, c in enumerate(BUSI_CLASSES)}
BUSI_N_CLASSES  = 3

# Few-shot seeds & repeats
N_SHOT_SEEDS  = 5     # repeat each N-shot experiment 5 times → mean±std
FINETUNE_EPOCHS      = 45
FINETUNE_WARMUP      = 3
FINETUNE_BATCH       = 16
FINETUNE_LR          = 5e-5
FINETUNE_WEIGHT_DECAY = 1e-4

# Magnification: BUSI is ultrasound — no magnification concept;
# we map all to mag_label = 0 (dummy)
BUSI_MAG_LABEL = 0

BUSI_PAPER_DPI = 150

# Plot style (match main notebook)
SKY_BLUE   = "#87CEEB"
DEEP_BLUE  = "#1E90FF"
NAVY       = "#1a3a5c"
LIGHT_GRAY = "#f0f4f8"

plt.rcParams.update({
    "axes.facecolor":   LIGHT_GRAY,
    "figure.facecolor": "white",
    "axes.edgecolor":   NAVY,
    "axes.labelcolor":  NAVY,
    "xtick.color":      NAVY,
    "ytick.color":      NAVY,
    "text.color":       NAVY,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "DejaVu Sans",
})

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


# ═══════════════════════════════════════════════════════════════════════════
# BUSI DATASET BUILDER
# ═══════════════════════════════════════════════════════════════════════════
def build_busi_df(busi_root: Path) -> pd.DataFrame:
    """
    Scan BUSI directory structure:
        busi_root/benign/*.png
        busi_root/malignant/*.png
        busi_root/normal/*.png
    Excludes *_mask.png files (GT masks).
    Returns DataFrame with columns: path, class_name, label, mag_label.
    """
    rows = []
    for cls_name in BUSI_CLASSES:
        cls_dir = busi_root / cls_name
        if not cls_dir.exists():
            print(f"  ⚠️  Directory not found: {cls_dir}")
            continue
        imgs = sorted([
            p for p in cls_dir.glob("*.png")
            if "_mask" not in p.name
        ])
        for p in imgs:
            rows.append({
                "path":       str(p),
                "class_name": cls_name,
                "label":      BUSI_CLASS2ID[cls_name],
                "mag_label":  BUSI_MAG_LABEL,
            })
    df = pd.DataFrame(rows)
    print(f"\n  📊 BUSI Dataset:")
    for cls_name in BUSI_CLASSES:
        n = (df["class_name"] == cls_name).sum()
        print(f"     {cls_name:12s} : {n:4d} images")
    print(f"     {'TOTAL':12s} : {len(df):4d} images")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# NMT CACHE FOR BUSI
# ═══════════════════════════════════════════════════════════════════════════
class BUSINMTCache:
    """
    Lightweight NMT cache for BUSI — stores tokens and heatmaps only.
    """
    def __init__(self, df: pd.DataFrame, ext, tag: str = "busi"):
        cache_path = BUSI_OUT / f"nmt_cache_{tag}.npz"
        if cache_path.exists():
            print(f"  📦 Loading BUSI NMT cache: {cache_path.name}")
            data           = np.load(str(cache_path))
            self.tokens    = data["tokens"]
            self.heatmaps  = data["heatmaps"]
            self.sigs      = data["sigs"]
            self.stains    = data["stains"]
        else:
            print(f"  ⏳ Precomputing BUSI NMT for {len(df)} images …")
            tok_l, heat_l, sig_l, stain_l = [], [], [], []
            for i, (_, row) in enumerate(df.iterrows()):
                if (i + 1) % 200 == 0 or (i + 1) == len(df):
                    print(f"     [{i+1}/{len(df)}]")
                try:
                    img = Image.open(row["path"]).convert("RGB")
                    tok, heat, sig, stain = ext.extract(img)
                except Exception:
                    tok   = np.zeros((ext.n_tokens, NMT_FEAT_DIM),  dtype=np.float32)
                    heat  = np.ones(ext.n_patches,                   dtype=np.float32) / ext.n_patches
                    sig   = np.zeros(NMT_SIG_DIM,                   dtype=np.float32)
                    stain = np.zeros(STAIN_FEAT_DIM,                 dtype=np.float32)
                tok_l.append(tok); heat_l.append(heat)
                sig_l.append(sig); stain_l.append(stain)
            self.tokens   = np.stack(tok_l).astype(np.float32)
            self.heatmaps = np.stack(heat_l).astype(np.float32)
            self.sigs     = np.stack(sig_l).astype(np.float32)
            self.stains   = np.stack(stain_l).astype(np.float32)
            np.savez_compressed(str(cache_path),
                                tokens=self.tokens, heatmaps=self.heatmaps,
                                sigs=self.sigs, stains=self.stains)
            print(f"  ✅ BUSI NMT cache saved → {cache_path.name}")

    def __getitem__(self, idx):
        return (self.tokens[idx], self.heatmaps[idx],
                self.sigs[idx], self.stains[idx])

    def subset(self, indices):
        """Return a SubCache view over the given indices."""
        return _SubBUSICache(self, np.array(indices))


class _SubBUSICache:
    def __init__(self, parent, indices):
        self.parent  = parent
        self.indices = indices
    def __getitem__(self, i):
        return self.parent[self.indices[i]]


# ═══════════════════════════════════════════════════════════════════════════
# BUSI PYTORCH DATASET
# ═══════════════════════════════════════════════════════════════════════════
class BUSIDataset(Dataset):
    def __init__(self, df, nmt_cache, transform=None):
        self.df        = df.reset_index(drop=True)
        self.nmt_cache = nmt_cache
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        tok, heat, sig, stain = self.nmt_cache[idx]
        return (img,
                int(row["label"]),
                int(row["label"]),          # binary_label (reuse)
                int(row["mag_label"]),
                torch.from_numpy(tok),
                torch.from_numpy(heat),
                torch.from_numpy(sig),
                torch.from_numpy(stain))


# ═══════════════════════════════════════════════════════════════════════════
# BUSI-ADAPTED MODEL
# ═══════════════════════════════════════════════════════════════════════════
class BUSIAdaptedModel(nn.Module):
    """
    Wraps the pretrained CausalMorphViT backbone and replaces the
    classification heads with new 3-class heads for BUSI.

    Zero-shot: run forward with frozen backbone, map 8-class logits to 3.
    Few-shot / Fine-tune: train only the new heads (or full model).
    """
    def __init__(self, ckpt_path: Path, n_classes: int = BUSI_N_CLASSES,
                 freeze_backbone: bool = True):
        super().__init__()

        # ── Load pretrained backbone ──────────────────────────────────────
        backbone = CausalMorphViT(num_subclasses=8).to(DEVICE)
        backbone.load_state_dict(
            torch.load(str(ckpt_path), map_location=DEVICE))
        print(f"  ✅ Loaded checkpoint: {ckpt_path.name}")

        # Keep everything except the final classification heads
        self.vit              = backbone.vit
        self.nmt_prior_alpha  = backbone.nmt_prior_alpha
        self.nmt_proj         = backbone.nmt_proj
        self.nmt_pos          = backbone.nmt_pos
        self.mc_moe           = backbone.mc_moe
        self.grl              = backbone.grl
        self.stain_disc       = backbone.stain_disc
        self.projector        = backbone.projector

        dim = backbone.vit.num_features   # 192

        # ── New 3-class heads ─────────────────────────────────────────────
        self.head_busi = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(dim, n_classes)
        )
        # Keep a 2-class head for binary (benign vs non-benign) — useful for
        # AUC metrics in 3-class setting
        self.head_bin2 = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(dim, 2)
        )

        # ── Freeze / unfreeze ─────────────────────────────────────────────
        if freeze_backbone:
            self._freeze_backbone()
        else:
            self._unfreeze_all()

    def _freeze_backbone(self):
        for name, p in self.named_parameters():
            if "head_busi" in name or "head_bin2" in name:
                p.requires_grad = True
            else:
                p.requires_grad = False

    def _unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

    @staticmethod
    def _pos_drop(vit, x):
        return vit.pos_drop(x) if hasattr(vit, "pos_drop") else x

    def get_features(self, x, mag, nmt_tokens, nmt_heatmap):
        B   = x.shape[0]
        vit = self.vit
        patches = vit.patch_embed(x)
        alpha   = self.nmt_prior_alpha.clamp(0.0, 2.0)
        patches = patches * (1.0 + alpha * nmt_heatmap.unsqueeze(-1))
        cls_tok = vit.cls_token.expand(B, -1, -1)
        x_seq   = torch.cat([cls_tok, patches], dim=1) + vit.pos_embed
        x_seq   = self._pos_drop(vit, x_seq)
        n_tok   = self.nmt_proj(nmt_tokens) + self.nmt_pos
        x_seq   = torch.cat([x_seq, n_tok], dim=1)
        for blk in vit.blocks:
            x_seq = blk(x_seq)
        x_seq = vit.norm(x_seq)
        feat  = x_seq[:, 0]
        feat_moe, moe_loss = self.mc_moe(feat, mag)
        return feat_moe, moe_loss

    def forward(self, x, mag, nmt_tokens, nmt_heatmap, return_feat=False):
        feat_moe, moe_loss = self.get_features(x, mag, nmt_tokens, nmt_heatmap)
        out_busi = self.head_busi(feat_moe)
        out_bin2 = self.head_bin2(feat_moe)
        stain_pred = self.stain_disc(self.grl(feat_moe))
        if return_feat:
            z = F.normalize(self.projector(feat_moe), dim=1)
            return out_busi, out_bin2, z, feat_moe, moe_loss, stain_pred
        return out_busi, out_bin2, moe_loss, stain_pred


# ═══════════════════════════════════════════════════════════════════════════
# ZERO-SHOT: MAP 8-CLASS LOGITS → 3-CLASS
# ═══════════════════════════════════════════════════════════════════════════
# BreakHis subclass → BUSI mapping heuristic:
#   benign     (0-3) → BUSI "benign"    (label 0)
#   malignant  (4-7) → BUSI "malignant" (label 1)
#   normal     (none in BreakHis) → BUSI "normal" (label 2)
#
# For zero-shot we use the pretrained 8-class head logits:
#   BUSI_benign score    = mean of BreakHis classes 0-3
#   BUSI_malignant score = mean of BreakHis classes 4-7
#   BUSI_normal score    = -(BUSI_benign + BUSI_malignant) / 2
#   (or a small fixed prior — empirically works as a weak baseline)

BREAKHIS_TO_BUSI = {
    0: 0, 1: 0, 2: 0, 3: 0,   # benign  → 0
    4: 1, 5: 1, 6: 1, 7: 1,   # malignant → 1
}


def zero_shot_logits(logits_8: torch.Tensor) -> torch.Tensor:
    """
    Map 8-class BreakHis logits → 3-class BUSI logits.
    B x 8 → B x 3
    """
    prob8 = F.softmax(logits_8, dim=1)
    beni  = prob8[:, 0:4].sum(1, keepdim=True)     # benign
    mali  = prob8[:, 4:8].sum(1, keepdim=True)     # malignant
    norm  = 1.0 - beni - mali                       # normal (residual)
    return torch.cat([beni, mali, norm], dim=1)


# ═══════════════════════════════════════════════════════════════════════════
# METRIC HELPERS
# ═══════════════════════════════════════════════════════════════════════════
def compute_busi_metrics(y_true, y_pred, y_prob, n_classes=BUSI_N_CLASSES,
                         class_names=None):
    """
    Full metrics for BUSI 3-class task.
    """
    if class_names is None:
        class_names = BUSI_CLASSES

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc  = float((y_pred == y_true).mean())
    bacc = float(balanced_accuracy_score(y_true, y_pred))
    prec = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    rec  = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    f1   = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    mcc  = float(matthews_corrcoef(y_true, y_pred))
    kap  = float(cohen_kappa_score(y_true, y_pred))

    # G-mean
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    per_rec = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    gmean = float(np.prod(per_rec) ** (1.0 / n_classes))

    # Specificity (macro)
    spec_list = []
    for c in range(n_classes):
        tp = cm[c, c]; fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp; tn = cm.sum() - tp - fn - fp
        spec_list.append(tn / (tn + fp + 1e-8))
    spec = float(np.mean(spec_list))

    # AUC (OvR macro)
    try:
        y_bin = label_binarize(y_true, classes=list(range(n_classes)))
        auc_val = float(roc_auc_score(
            y_bin, y_prob[:, :n_classes], multi_class="ovr", average="macro"))
    except Exception:
        auc_val = float("nan")

    return dict(acc=acc, bacc=bacc, auc=auc_val, f1=f1,
                sens=rec, spec=spec, prec=prec,
                mcc=mcc, kappa=kap, gmean=gmean)


METRIC_KEYS = ["acc", "bacc", "auc", "f1", "sens", "spec",
               "prec", "mcc", "kappa", "gmean"]


def print_metrics(metrics: dict, prefix: str = ""):
    print(f"\n  {prefix}")
    for k in METRIC_KEYS:
        v = metrics.get(k, float("nan"))
        print(f"    {k:10s}: {v:.4f}")


# ═══════════════════════════════════════════════════════════════════════════
# PLOT HELPERS
# ═══════════════════════════════════════════════════════════════════════════
def plot_cm(cm, title, save_path, labels=BUSI_CLASSES):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=np.arange(len(labels)), yticks=np.arange(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel="Predicted", ylabel="True", title=title)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else NAVY, fontsize=9)
    fig.tight_layout()
    plt.savefig(str(save_path), dpi=BUSI_PAPER_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 {Path(save_path).name}")


def plot_metrics_bar(results_dict: dict, title: str, save_path: str,
                     keys=None):
    """
    Bar chart comparing methods across metrics.
    results_dict: {method_name: metrics_dict}
    """
    if keys is None:
        keys = ["acc", "bacc", "auc", "f1", "sens", "spec", "mcc", "gmean"]
    methods = list(results_dict.keys())
    n_metrics = len(keys)
    n_methods = len(methods)
    x = np.arange(n_metrics)
    width = 0.8 / n_methods
    cmap  = plt.cm.get_cmap("tab10", n_methods)

    fig, ax = plt.subplots(figsize=(max(10, n_metrics * 1.4), 5))
    for i, method in enumerate(methods):
        vals = [results_dict[method].get(k, 0.0) for k in keys]
        bars = ax.bar(x + i * width - (n_methods - 1) * width / 2,
                      vals, width * 0.9, label=method, color=cmap(i),
                      edgecolor=NAVY, linewidth=0.6)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{v:.3f}", ha="center", va="bottom",
                    fontsize=5.5, color=NAVY)

    ax.set_xticks(x); ax.set_xticklabels([k.upper() for k in keys], fontsize=8)
    ax.set_ylim([0, 1.08]); ax.set_ylabel("Score")
    ax.set_title(title, color=NAVY)
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    plt.savefig(str(save_path), dpi=BUSI_PAPER_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 {Path(save_path).name}")


def plot_training_history(history: list, title: str, save_path: str):
    df_h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(df_h["epoch"], df_h["train_loss"], color=DEEP_BLUE, label="Train")
    axes[0].plot(df_h["epoch"], df_h["val_loss"],   color=SKY_BLUE,  label="Val", ls="--")
    axes[0].set_title(f"Loss — {title}", color=NAVY)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
    axes[1].plot(df_h["epoch"], df_h["train_acc"], color=DEEP_BLUE, label="Train")
    axes[1].plot(df_h["epoch"], df_h["val_acc"],   color=SKY_BLUE,  label="Val", ls="--")
    axes[1].set_title(f"Accuracy — {title}", color=NAVY)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()
    fig.tight_layout()
    plt.savefig(str(save_path), dpi=BUSI_PAPER_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 {Path(save_path).name}")


def plot_summary_table(results_dict: dict, title: str, save_path: str):
    """
    Renders a neat table image of all methods × metrics.
    """
    methods = list(results_dict.keys())
    keys    = METRIC_KEYS
    cell_data  = []
    col_labels = ["Method"] + [k.upper() for k in keys]
    for method in methods:
        m = results_dict[method]
        row = [method] + [f"{m.get(k, float('nan')):.4f}" for k in keys]
        cell_data.append(row)

    fig, ax = plt.subplots(figsize=(len(col_labels) * 1.3, len(methods) * 0.6 + 1.5))
    ax.axis("off")
    tbl = ax.table(cellText=cell_data, colLabels=col_labels,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8)
    tbl.scale(1.0, 1.7)
    for j in range(len(col_labels)):
        tbl[0, j].set_facecolor(SKY_BLUE)
        tbl[0, j].set_text_props(color=NAVY, fontweight="bold")
    for i in range(len(methods)):
        tbl[i+1, 0].set_facecolor(LIGHT_GRAY)
    ax.set_title(title, color=NAVY, fontsize=10, pad=12)
    fig.tight_layout()
    plt.savefig(str(save_path), dpi=BUSI_PAPER_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"   💾 {Path(save_path).name}")


# ═══════════════════════════════════════════════════════════════════════════
# INFERENCE HELPER
# ═══════════════════════════════════════════════════════════════════════════
def _to(t): return t.to(DEVICE, non_blocking=True)


def make_loader(df, nmt_cache, transform, batch_size=32, shuffle=False):
    ds = BUSIDataset(df, nmt_cache, transform)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True,
                      persistent_workers=False, drop_last=False)


@torch.no_grad()
def collect_preds_busi(loader, model):
    """
    Returns: preds (N,), probs (N, C), labels (N,)
    """
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    for x, y, _, y_mag, n_tok, n_heat, _, _ in loader:
        x      = _to(x); y = _to(y)
        y_mag  = _to(y_mag)
        n_tok  = _to(n_tok); n_heat = _to(n_heat)
        with autocast(device_type="cuda"):
            out_busi, _, _, _ = model(x, y_mag, n_tok, n_heat)
        all_preds.extend(out_busi.argmax(1).cpu().numpy())
        all_probs.extend(F.softmax(out_busi, dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return (np.array(all_preds),
            np.array(all_probs),
            np.array(all_labels))


@torch.no_grad()
def collect_preds_zeroshot(loader, backbone_8class):
    """
    Zero-shot: use raw CausalMorphViT (8-class) and map logits to BUSI 3-class.
    """
    backbone_8class.eval()
    all_preds, all_probs, all_labels = [], [], []
    for x, y, _, y_mag, n_tok, n_heat, _, _ in loader:
        x      = _to(x); y = _to(y)
        y_mag  = _to(y_mag)
        n_tok  = _to(n_tok); n_heat = _to(n_heat)
        with autocast(device_type="cuda"):
            out_sub, out_bin, _, _ = backbone_8class(x, y_mag, n_tok, n_heat)
        prob3 = zero_shot_logits(out_sub)   # B x 3
        all_preds.extend(prob3.argmax(1).cpu().numpy())
        all_probs.extend(prob3.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    return (np.array(all_preds),
            np.array(all_probs),
            np.array(all_labels))


# ═══════════════════════════════════════════════════════════════════════════
# FEW-SHOT SUPPORT-SET SAMPLING
# ═══════════════════════════════════════════════════════════════════════════
def sample_support_set(df, n_shot: int, seed: int, label_col="label"):
    """
    Sample n_shot examples per class (support set).
    Returns (support_df, query_df).
    """
    rng = random.Random(seed)
    support_idx, query_idx = [], []
    for cls_id in sorted(df[label_col].unique()):
        idx = df[df[label_col] == cls_id].index.tolist()
        rng.shuffle(idx)
        support_idx.extend(idx[:n_shot])
        query_idx.extend(idx[n_shot:])
    support_df = df.loc[support_idx].reset_index(drop=True)
    query_df   = df.loc[query_idx].reset_index(drop=True)
    return support_df, query_df


# ═══════════════════════════════════════════════════════════════════════════
# PROTOTYPE-BASED FEW-SHOT CLASSIFICATION
# ═══════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def few_shot_prototypical(support_df, query_df,
                          full_cache, model: BUSIAdaptedModel,
                          support_cache_indices, query_cache_indices):
    """
    1. Embed support set → compute per-class prototypes.
    2. Classify query set by nearest prototype (cosine distance).
    """
    model.eval()

    # Support embeddings
    sup_loader = make_loader(
        support_df,
        full_cache.subset(support_cache_indices),
        val_tf, batch_size=32)

    sup_feats, sup_labels = [], []
    for x, y, _, y_mag, n_tok, n_heat, _, _ in sup_loader:
        x = _to(x); y = _to(y)
        y_mag = _to(y_mag); n_tok = _to(n_tok); n_heat = _to(n_heat)
        with autocast(device_type="cuda"):
            feat, _ = model.get_features(x, y_mag, n_tok, n_heat)
        sup_feats.append(feat.cpu()); sup_labels.extend(y.cpu().numpy())
    sup_feats  = torch.cat(sup_feats, dim=0)
    sup_labels = np.array(sup_labels)

    # Prototypes
    protos = {}
    for cls_id in range(BUSI_N_CLASSES):
        m = sup_labels == cls_id
        if m.sum() > 0:
            protos[cls_id] = F.normalize(sup_feats[m].mean(0), dim=0)
        else:
            protos[cls_id] = torch.zeros(sup_feats.shape[-1])
    proto_mat = torch.stack([protos[c] for c in range(BUSI_N_CLASSES)])  # (3, D)

    # Query embeddings
    qry_loader = make_loader(
        query_df,
        full_cache.subset(query_cache_indices),
        val_tf, batch_size=32)

    all_preds, all_probs, all_labels = [], [], []
    for x, y, _, y_mag, n_tok, n_heat, _, _ in qry_loader:
        x = _to(x); y_mag = _to(y_mag)
        n_tok = _to(n_tok); n_heat = _to(n_heat)
        with autocast(device_type="cuda"):
            feat, _ = model.get_features(x, y_mag, n_tok, n_heat)
        feat_n = F.normalize(feat.cpu(), dim=1)   # (B, D)
        sims   = feat_n @ proto_mat.T              # (B, 3)
        probs  = F.softmax(sims * 10, dim=1)       # temperature scaling
        preds  = probs.argmax(1)
        all_preds.extend(preds.numpy())
        all_probs.extend(probs.numpy())
        all_labels.extend(y.cpu().numpy())

    return (np.array(all_preds),
            np.array(all_probs),
            np.array(all_labels))


# ═══════════════════════════════════════════════════════════════════════════
# FINE-TUNING
# ═══════════════════════════════════════════════════════════════════════════
def finetune_on_busi(train_df, val_df, test_df,
                     train_cache_indices, val_cache_indices, test_cache_indices,
                     full_cache,
                     freeze_backbone: bool = False,
                     tag: str = "finetune"):
    """
    Fine-tune BUSIAdaptedModel on BUSI train split.
    Returns test metrics after fine-tuning.
    """
    model = BUSIAdaptedModel(CKPT_PATH, freeze_backbone=freeze_backbone).to(DEVICE)
    model_tag = "head-only" if freeze_backbone else "full"
    print(f"\n  🔧 Fine-tuning [{model_tag}] for {FINETUNE_EPOCHS} epochs …")
    print(f"     Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

    train_cache = full_cache.subset(train_cache_indices)
    val_cache   = full_cache.subset(val_cache_indices)
    test_cache  = full_cache.subset(test_cache_indices)

    # Class weights
    w = compute_class_weight("balanced",
                             classes=np.arange(BUSI_N_CLASSES),
                             y=train_df["label"].values)
    class_weights = torch.tensor(w, dtype=torch.float).to(DEVICE)
    ce_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.10)

    train_loader = make_loader(train_df, train_cache, train_tf,
                               FINETUNE_BATCH, shuffle=True)
    val_loader   = make_loader(val_df,   val_cache,   val_tf,   32)
    test_loader  = make_loader(test_df,  test_cache,  val_tf,   32)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=FINETUNE_LR, weight_decay=FINETUNE_WEIGHT_DECAY)

    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0,
        total_iters=FINETUNE_WARMUP)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(FINETUNE_EPOCHS - FINETUNE_WARMUP, 1))
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, [warmup_sched, cosine_sched],
        milestones=[FINETUNE_WARMUP])
    scaler = GradScaler(device="cuda")

    best_val_acc = 0.0
    best_state   = None
    history      = []

    for epoch in range(FINETUNE_EPOCHS):
        model.train()
        tr_loss = tr_correct = tr_n = 0
        for x, y, _, y_mag, n_tok, n_heat, _, _ in train_loader:
            x = _to(x); y = _to(y); y_mag = _to(y_mag)
            n_tok = _to(n_tok); n_heat = _to(n_heat)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda"):
                out_busi, _, _, _ = model(x, y_mag, n_tok, n_heat)
                loss = ce_fn(out_busi, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            tr_loss    += loss.item() * len(y)
            tr_correct += (out_busi.argmax(1) == y).sum().item()
            tr_n       += len(y)
        scheduler.step()

        # Val
        model.eval()
        val_loss = val_correct = val_n = 0
        with torch.no_grad():
            for x, y, _, y_mag, n_tok, n_heat, _, _ in val_loader:
                x = _to(x); y = _to(y); y_mag = _to(y_mag)
                n_tok = _to(n_tok); n_heat = _to(n_heat)
                with autocast(device_type="cuda"):
                    out_busi, _, _, _ = model(x, y_mag, n_tok, n_heat)
                    loss = ce_fn(out_busi, y)
                val_loss    += loss.item() * len(y)
                val_correct += (out_busi.argmax(1) == y).sum().item()
                val_n       += len(y)

        tr_acc  = tr_correct  / max(tr_n, 1)
        val_acc = val_correct / max(val_n, 1)
        tr_loss_avg  = tr_loss  / max(tr_n, 1)
        val_loss_avg = val_loss / max(val_n, 1)

        history.append(dict(epoch=epoch+1,
                            train_loss=tr_loss_avg, val_loss=val_loss_avg,
                            train_acc=tr_acc,        val_acc=val_acc))

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"     Ep {epoch+1:02d}/{FINETUNE_EPOCHS}"
                  f" | TrLoss={tr_loss_avg:.4f} TrAcc={tr_acc:.4f}"
                  f" | ValLoss={val_loss_avg:.4f} ValAcc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = copy.deepcopy(model.state_dict())

    # Load best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  ✅ Best Val Acc: {best_val_acc:.4f}")

    # Save model
    ft_ckpt = BUSI_OUT / f"busi_finetuned_{tag}.pth"
    torch.save(model.state_dict(), str(ft_ckpt))
    print(f"  💾 Fine-tuned model saved: {ft_ckpt.name}")

    # Test
    preds, probs, labels = collect_preds_busi(test_loader, model)
    metrics = compute_busi_metrics(labels, preds, probs)

    # Confusion matrix
    cm = confusion_matrix(labels, preds, labels=list(range(BUSI_N_CLASSES)))
    plot_cm(cm, f"CM — Fine-tuned [{model_tag}]",
            BUSI_OUT / f"cm_finetune_{tag}.png")

    # Training history plot
    plot_training_history(history, f"Fine-tune [{model_tag}]",
                          BUSI_OUT / f"history_finetune_{tag}.png")

    del model, optimizer, scaler
    torch.cuda.empty_cache()

    return metrics, history


# ═══════════════════════════════════════════════════════════════════════════
# TRAIN / VAL / TEST SPLIT FOR BUSI
# ═══════════════════════════════════════════════════════════════════════════
def stratified_split_busi(df, train_ratio=0.70, val_ratio=0.15,
                           seed=SEED):
    """
    Stratified split by label.
    Returns (train_df, val_df, test_df) with original DF indices reset.
    Also returns the index arrays for cache subsetting.
    """
    rng = random.Random(seed)
    train_idx = []; val_idx = []; test_idx = []
    for cls_id in sorted(df["label"].unique()):
        idx = df[df["label"] == cls_id].index.tolist()
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        train_idx.extend(idx[:n_train])
        val_idx.extend(idx[n_train:n_train + n_val])
        test_idx.extend(idx[n_train + n_val:])

    train_df = df.loc[train_idx].reset_index(drop=True)
    val_df   = df.loc[val_idx].reset_index(drop=True)
    test_df  = df.loc[test_idx].reset_index(drop=True)

    # cache indices: position in original df
    train_cache_idx = np.array(train_idx)
    val_cache_idx   = np.array(val_idx)
    test_cache_idx  = np.array(test_idx)

    print(f"\n  📊 BUSI Split:")
    for name, sdf in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        dist = sdf["label"].value_counts().sort_index()
        dstr = "  ".join([f"{BUSI_CLASSES[k]}: {v}" for k, v in dist.items()])
        print(f"     {name:6s} ({len(sdf):4d}) → {dstr}")

    return (train_df, val_df, test_df,
            train_cache_idx, val_cache_idx, test_cache_idx)


# ═══════════════════════════════════════════════════════════════════════════
# MAIN EVALUATION PIPELINE
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'█'*65}")
print(f"  BUSI External Dataset Evaluation — CausalMorph-ViT")
print(f"  Checkpoint : {CKPT_PATH.name}")
print(f"  Output dir : {BUSI_OUT}")
print(f"{'█'*65}")

# ── 1. Build BUSI dataframe ────────────────────────────────────────────────
df_busi = build_busi_df(BUSI_ROOT)

# ── 2. Precompute NMT cache for ALL BUSI images ────────────────────────────
print("\n📐 Building NMT cache for all BUSI images …")
full_cache = BUSINMTCache(df_busi, _nmt_ext, tag="all")

# ── 3. Stratified split ────────────────────────────────────────────────────
(train_df, val_df, test_df,
 train_cache_idx, val_cache_idx, test_cache_idx) = stratified_split_busi(df_busi)

test_cache  = full_cache.subset(test_cache_idx)
test_loader = make_loader(test_df, test_cache, val_tf, batch_size=32)

# ── 4. Load pretrained backbone (original 8-class) ────────────────────────
print(f"\n{'─'*65}")
print(f"  [ZERO-SHOT] Evaluation")
print(f"{'─'*65}")

backbone_8 = CausalMorphViT(num_subclasses=8).to(DEVICE)
backbone_8.load_state_dict(torch.load(str(CKPT_PATH), map_location=DEVICE))
backbone_8.eval()
print(f"  ✅ Backbone loaded")

all_cache_loader = make_loader(df_busi, full_cache,
                               val_tf, batch_size=32)
test_full_loader = make_loader(
    test_df, full_cache.subset(test_cache_idx), val_tf, batch_size=32)

zs_preds, zs_probs, zs_labels = collect_preds_zeroshot(
    test_full_loader, backbone_8)
zs_metrics = compute_busi_metrics(zs_labels, zs_preds, zs_probs)
print_metrics(zs_metrics, "Zero-Shot Results (Test Set)")

cm_zs = confusion_matrix(zs_labels, zs_preds,
                         labels=list(range(BUSI_N_CLASSES)))
plot_cm(cm_zs, "Confusion Matrix — Zero-Shot",
        BUSI_OUT / "cm_zeroshot.png")

del backbone_8
torch.cuda.empty_cache()

# Collect all results for final comparison
ALL_RESULTS = {"Zero-Shot": zs_metrics}


# ── 5. Few-Shot (1-shot and 5-shot) ───────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  [FEW-SHOT] Prototypical Classification")
print(f"{'─'*65}")

# We build a BUSIAdaptedModel with frozen backbone for feature extraction
print("\n  Building adapter model for few-shot embedding …")
fewshot_model = BUSIAdaptedModel(CKPT_PATH, freeze_backbone=True).to(DEVICE)
fewshot_model.eval()

for n_shot in [1, 5]:
    shot_metrics_list = []
    print(f"\n  ── {n_shot}-Shot (×{N_SHOT_SEEDS} random seeds) ──")
    for seed_i in range(N_SHOT_SEEDS):
        seed_val = SEED + seed_i * 13

        # Sample support set from TEST set only (true few-shot setting)
        support_df, query_df = sample_support_set(
            test_df.copy(), n_shot=n_shot, seed=seed_val)

        # Map back to full_cache indices
        # support_df / query_df have positions within test_df
        # test_df positions → test_cache_idx → full_cache positions
        sup_global_idx  = test_cache_idx[support_df.index.tolist()
                                         if len(support_df) > 0
                                         else []]
        qry_global_idx  = test_cache_idx[query_df.index.tolist()
                                         if len(query_df) > 0
                                         else []]

        if len(query_df) == 0:
            print(f"     Seed {seed_i}: query set empty — skip")
            continue

        preds, probs, labels = few_shot_prototypical(
            support_df, query_df,
            full_cache,
            fewshot_model,
            support_cache_indices=sup_global_idx,
            query_cache_indices=qry_global_idx,
        )
        m = compute_busi_metrics(labels, preds, probs)
        shot_metrics_list.append(m)
        print(f"     Seed {seed_i:2d} | "
              f"Acc={m['acc']:.4f}  AUC={m['auc']:.4f}  "
              f"F1={m['f1']:.4f}  MCC={m['mcc']:.4f}")

    if shot_metrics_list:
        # Aggregate mean±std over seeds
        agg = {}
        for k in METRIC_KEYS:
            vals = [m[k] for m in shot_metrics_list if not np.isnan(m[k])]
            agg[k] = float(np.mean(vals)) if vals else float("nan")

        mean_std_str = "  ".join([
            f"{k}={agg[k]:.4f}" for k in ["acc", "auc", "f1", "mcc", "gmean"]])
        print(f"\n  {n_shot}-Shot MEAN → {mean_std_str}")

        ALL_RESULTS[f"{n_shot}-Shot"] = agg

        # Confusion matrix using last seed's run
        cm_fs = confusion_matrix(labels, preds,
                                 labels=list(range(BUSI_N_CLASSES)))
        plot_cm(cm_fs, f"CM — {n_shot}-Shot (last seed)",
                BUSI_OUT / f"cm_{n_shot}shot.png")

del fewshot_model
torch.cuda.empty_cache()


# ── 6. Fine-Tuning: heads only ─────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  [FINE-TUNE] Heads Only (backbone frozen)")
print(f"{'─'*65}")

ft_head_metrics, ft_head_history = finetune_on_busi(
    train_df, val_df, test_df,
    train_cache_idx, val_cache_idx, test_cache_idx,
    full_cache,
    freeze_backbone=True,
    tag="heads_only")

print_metrics(ft_head_metrics, "Fine-Tune [Heads Only] — Test Results")
ALL_RESULTS["FineTune-HeadsOnly"] = ft_head_metrics


# ── 7. Fine-Tuning: full model ─────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  [FINE-TUNE] Full Model (all layers trainable)")
print(f"{'─'*65}")

ft_full_metrics, ft_full_history = finetune_on_busi(
    train_df, val_df, test_df,
    train_cache_idx, val_cache_idx, test_cache_idx,
    full_cache,
    freeze_backbone=False,
    tag="full_model")

print_metrics(ft_full_metrics, "Fine-Tune [Full Model] — Test Results")
ALL_RESULTS["FineTune-FullModel"] = ft_full_metrics


# ═══════════════════════════════════════════════════════════════════════════
# FINAL COMPARISON PLOTS & TABLES
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*65}")
print(f"  📊 Generating comparison plots …")
print(f"{'─'*65}")

plot_metrics_bar(
    ALL_RESULTS,
    "BUSI Evaluation — Method Comparison",
    str(BUSI_OUT / "comparison_bar.png"))

plot_summary_table(
    ALL_RESULTS,
    "BUSI Evaluation Summary — CausalMorph-ViT",
    str(BUSI_OUT / "summary_table.png"))

# ── Save CSV ───────────────────────────────────────────────────────────────
rows = []
for method, m in ALL_RESULTS.items():
    row = {"method": method}
    row.update({k: m.get(k, float("nan")) for k in METRIC_KEYS})
    rows.append(row)
csv_path = BUSI_OUT / "busi_all_results.csv"
pd.DataFrame(rows).to_csv(str(csv_path), index=False)
print(f"  💾 Results CSV: {csv_path.name}")

# ═══════════════════════════════════════════════════════════════════════════
# FINAL PRINT SUMMARY
# ═══════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*65}")
print(f"  📊 BUSI FINAL RESULTS SUMMARY")
print(f"{'═'*65}")
header = f"  {'Method':<25s}" + "".join([f" {k.upper():>7s}" for k in METRIC_KEYS])
print(header)
print(f"  {'─'*25}" + "─" * (8 * len(METRIC_KEYS)))
for method, m in ALL_RESULTS.items():
    vals = "".join([f" {m.get(k, float('nan')):>7.4f}" for k in METRIC_KEYS])
    print(f"  {method:<25s}{vals}")
print(f"{'═'*65}")

print(f"\n💾 All outputs saved to: {BUSI_OUT}/")
saved = [
    "busi_all_results.csv",
    "comparison_bar.png",
    "summary_table.png",
    "cm_zeroshot.png",
    "cm_1shot.png",
    "cm_5shot.png",
    "cm_finetune_heads_only.png",
    "cm_finetune_full_model.png",
    "history_finetune_heads_only.png",
    "history_finetune_full_model.png",
    "busi_finetuned_heads_only.pth",
    "busi_finetuned_full_model.pth",
]
for f in saved:
    print(f"   📄 {BUSI_OUT}/{f}")